# Model Experiment: DLinear

1. Target-only DLinear
2. DLinear-X: DLinear temporal backbone with future covariate and static feature adjustment

The goal is to understand whether a simple decomposition-linear model can forecast weekly Store-Dept sales, and whether adding Walmart-specific covariates improves performance.

DLinear decomposes the input sequence into:
- trend component
- residual / seasonal component

Then it applies linear projection from context length to prediction length.

For our task:
input: previous 52 weeks of Weekly_Sales
output: next 39 weeks of Weekly_Sales

Variant A: Target-only DLinear
- uses only past_target
- tests how much signal exists in historical sales alone

Variant B: DLinear-X
- uses DLinear as the temporal backbone
- adds future known covariates and static metadata through a correction head
- better adapted to Walmart because sales depend on holidays, markdowns, store size, type and department identity

## Setup and Imports

In [1]:
from pathlib import Path
import sys
import json
import random
from copy import deepcopy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [2]:
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import load_raw_data, last_n_weeks_split
from src.features import WalmartBasePreprocessor, WalmartNeuralPreprocessor
from src.datasets import (
    WalmartPrecomputedTrainingWindowDataset,
    WalmartPrecomputedForecastWindowDataset,
    FastTensorDataLoader,
)

In [3]:
DATA_DIR = repo_root / "data" / "raw"

CONTEXT_LENGTH = 52
PREDICTION_LENGTH = 39

BATCH_SIZE = 256
NUM_WORKERS = 0

SEED = 42

MLFLOW_EXPERIMENT_NAME = "DLinear_Training"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Device:", DEVICE)

Repo root: C:\Users\Myvari\Desktop\walmart-sales-forecasting
Data dir: C:\Users\Myvari\Desktop\walmart-sales-forecasting\data\raw
Device: cpu


In [4]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Dagshub/Mlflow initialization

In [5]:
pip install dagshub mlflow pandas matplotlib seaborn skops --quiet

Note: you may need to restart the kernel to use updated packages.


In [6]:
import dagshub
import mlflow

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='walmart-sales-forecasting', mlflow=True)

Accessing as myvari

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

## Load Dataset and Time Split

In [7]:
train, test, stores, features = load_raw_data(DATA_DIR)

print("train:", train.shape)
print("test:", test.shape)
print("stores:", stores.shape)
print("features:", features.shape)

print("\nTrain date range:")
print(train["Date"].min(), "->", train["Date"].max())

print("\nTest date range:")
print(test["Date"].min(), "->", test["Date"].max())

display(train.head())
display(test.head())
display(stores.head())
display(features.head())

train: (421570, 5)
test: (115064, 4)
stores: (45, 3)
features: (8190, 12)

Train date range:
2010-02-05 00:00:00 -> 2012-10-26 00:00:00

Test date range:
2012-11-02 00:00:00 -> 2013-07-26 00:00:00


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [8]:
assert {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}.issubset(train.columns)
assert {"Store", "Dept", "Date", "IsHoliday"}.issubset(test.columns)
assert {"Store", "Type", "Size"}.issubset(stores.columns)
assert {"Store", "Date", "IsHoliday"}.issubset(features.columns)

print("Raw data checks passed.")

Raw data checks passed.


In [9]:
train_raw_part, valid_raw_part = last_n_weeks_split(
    train,
    n_weeks=PREDICTION_LENGTH,
    date_col="Date",
)

print("train_raw_part:", train_raw_part.shape)
print("valid_raw_part:", valid_raw_part.shape)

print("\nTrain split date range:")
print(train_raw_part["Date"].min(), "->", train_raw_part["Date"].max())

print("\nValidation split date range:")
print(valid_raw_part["Date"].min(), "->", valid_raw_part["Date"].max())

print("\nUnique train dates:", train_raw_part["Date"].nunique())
print("Unique validation dates:", valid_raw_part["Date"].nunique())

train_raw_part: (305982, 5)
valid_raw_part: (115588, 5)

Train split date range:
2010-02-05 00:00:00 -> 2012-01-27 00:00:00

Validation split date range:
2012-02-03 00:00:00 -> 2012-10-26 00:00:00

Unique train dates: 104
Unique validation dates: 39


In [10]:
assert train_raw_part["Date"].max() < valid_raw_part["Date"].min()
assert valid_raw_part["Date"].nunique() == PREDICTION_LENGTH

print("Time split checks passed.")

Time split checks passed.


## Base + neural preprocessing

In [11]:
base_preprocessor = WalmartBasePreprocessor()
base_preprocessor.fit(stores, features)

train_base = base_preprocessor.transform(train_raw_part)
valid_base = base_preprocessor.transform(valid_raw_part)

print("train_base:", train_base.shape)
print("valid_base:", valid_base.shape)

assert len(train_base) == len(train_raw_part)
assert len(valid_base) == len(valid_raw_part)

display(train_base.head())

train_base: (305982, 42)
valid_base: (115588, 42)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,...,markdown_available_period,Year,Month,WeekOfYear,Week_sin,Week_cos,IsSuperBowl,IsLaborDay,IsThanksgiving,IsChristmas
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,...,0,2010,2,5,0.568065,0.822984,0,0,0,0
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,...,0,2010,2,6,0.663123,0.748511,1,0,0,0
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,...,0,2010,2,7,0.748511,0.663123,0,0,0,0
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,...,0,2010,2,8,0.822984,0.568065,0,0,0,0
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,...,0,2010,3,9,0.885456,0.464723,0,0,0,0


In [12]:
required_base_cols = [
    "Store", "Dept", "Date", "Weekly_Sales",
    "Type", "Size",
    "IsHoliday",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "MarkDown1_was_missing", "MarkDown2_was_missing", "MarkDown3_was_missing",
    "MarkDown4_was_missing", "MarkDown5_was_missing",
    "total_markdown", "abs_total_markdown",
    "positive_markdown_sum", "negative_markdown_sum",
    "markdown_missing_count", "markdown_available_period",
    "has_markdown_signal",
    "Week_sin", "Week_cos",
    "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas",
]

missing_train = [c for c in required_base_cols if c not in train_base.columns]
missing_valid = [c for c in required_base_cols if c not in valid_base.columns]

print("Missing in train_base:", missing_train)
print("Missing in valid_base:", missing_valid)

assert not missing_train
assert not missing_valid

print("Base preprocessing checks passed.")

Missing in train_base: []
Missing in valid_base: []
Base preprocessing checks passed.


In [13]:
neural_preprocessor = WalmartNeuralPreprocessor()
neural_preprocessor.fit(train_base)

train_panel = neural_preprocessor.transform(train_base)
valid_panel = neural_preprocessor.transform(valid_base)

dataset_cols = neural_preprocessor.get_dataset_columns()

print("train_panel:", train_panel.shape)
print("valid_panel:", valid_panel.shape)

print("\nDataset columns:")
for key, value in dataset_cols.items():
    print(f"{key}: {value}")

train_panel: (305982, 66)
valid_panel: (115588, 66)

Dataset columns:
target_col: Weekly_Sales_scaled
series_col: series_id
static_cat_cols: ['Store_id', 'Dept_id', 'Type_id']
static_real_cols: ['Size_scaled']
known_future_real_cols: ['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'MarkDown1_scaled', 'MarkDown2_scaled', 'MarkDown3_scaled', 'MarkDown4_scaled', 'MarkDown5_scaled', 'total_markdown_scaled', 'abs_total_markdown_scaled', 'positive_markdown_sum_scaled', 'negative_markdown_sum_scaled', 'markdown_missing_count_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'has_markdown_signal', 'markdown_available_period', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']


In [14]:
expected_neural_cols = [
    "series_id",
    "Store_id", "Dept_id", "Type_id",
    "Weekly_Sales_scaled",
    "target_mean", "target_std",
]

for col in expected_neural_cols:
    assert col in train_panel.columns, f"Missing from train_panel: {col}"
    assert col in valid_panel.columns, f"Missing from valid_panel: {col}"

assert train_panel["Weekly_Sales_scaled"].notna().all()
assert valid_panel["Weekly_Sales_scaled"].notna().all()

assert train_panel["target_mean"].notna().all()
assert valid_panel["target_mean"].notna().all()

assert train_panel["target_std"].notna().all()
assert valid_panel["target_std"].notna().all()

assert (train_panel["target_std"] > 0).all()
assert (valid_panel["target_std"] > 0).all()

print("Neural preprocessing checks passed.")

Neural preprocessing checks passed.


In [15]:
print("Static categorical columns:")
print(dataset_cols["static_cat_cols"])

print("\nStatic real columns:")
print(dataset_cols["static_real_cols"])

print("\nKnown future/time-varying feature columns:")
print("Count:", len(dataset_cols["known_future_real_cols"]))
print(dataset_cols["known_future_real_cols"])

Static categorical columns:
['Store_id', 'Dept_id', 'Type_id']

Static real columns:
['Size_scaled']

Known future/time-varying feature columns:
Count: 28
['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'MarkDown1_scaled', 'MarkDown2_scaled', 'MarkDown3_scaled', 'MarkDown4_scaled', 'MarkDown5_scaled', 'total_markdown_scaled', 'abs_total_markdown_scaled', 'positive_markdown_sum_scaled', 'negative_markdown_sum_scaled', 'markdown_missing_count_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'has_markdown_signal', 'markdown_available_period', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']


In [16]:
display(
    train_panel[
        ["Store", "Dept", "Type"]
        + dataset_cols["static_cat_cols"]
    ]
    .drop_duplicates()
    .head(20)
)

for col in dataset_cols["static_cat_cols"]:
    print(col)
    print("  train min/max:", train_panel[col].min(), train_panel[col].max())
    print("  valid min/max:", valid_panel[col].min(), valid_panel[col].max())
    print("  valid unknown count:", int((valid_panel[col] == 0).sum()))

,Store,Dept,Type,Store_id,Dept_id,Type_id
0,1,1,A,1,1,1
104,1,2,A,1,2,1
208,1,3,A,1,3,1
312,1,4,A,1,4,1
416,1,5,A,1,5,1
520,1,6,A,1,6,1
624,1,7,A,1,7,1
728,1,8,A,1,8,1
832,1,9,A,1,9,1
936,1,10,A,1,10,1


Store_id
  train min/max: 1 45
  valid min/max: 1 45
  valid unknown count: 0
Dept_id
  train min/max: 1 81
  valid min/max: 1 81
  valid unknown count: 0
Type_id
  train min/max: 1 3
  valid min/max: 1 3
  valid unknown count: 0


In [17]:
display(train_panel["Weekly_Sales_scaled"].describe())

series_scaled_summary = (
    train_panel
    .groupby("series_id")["Weekly_Sales_scaled"]
    .agg(["count", "mean", "std"])
    .reset_index()
)

display(series_scaled_summary.describe())

count    3.059820e+05
mean     1.223784e-17
std      9.945732e-01
min     -5.873652e+00
25%     -6.025369e-01
50%     -2.031418e-01
75%      4.276823e-01
max      9.767577e+00
Name: Weekly_Sales_scaled, dtype: float64

,count,mean,std
count,3306.000000,3.306000e+03,3269.000000
mean,92.553539,1.060251e-17,0.998470
std,28.794504,3.924025e-16,0.039085
min,1.000000,-2.013347e-15,0.000000
25%,104.000000,-1.220178e-16,1.000000
50%,104.000000,0.000000e+00,1.000000
75%,104.000000,1.301710e-16,1.000000
max,104.000000,3.269287e-15,1.000000


In [18]:
series_scaled_summary = (
    valid_panel
    .groupby("series_id")["Weekly_Sales_scaled"]
    .agg(["count", "mean", "std"])
    .reset_index()
)

display(series_scaled_summary.describe())

,count,mean,std
count,3204.000000,3204.000000,3151.000000
mean,36.076155,7.238062,5.206050
std,8.879486,407.298249,250.622157
min,1.000000,-9.146735,0.000000
25%,39.000000,-0.363890,0.469019
50%,39.000000,-0.048698,0.721007
75%,39.000000,0.360062,0.933382
max,39.000000,23054.672414,14069.079232


##  Dataset and DataLoaders

In [37]:
target_only_dataset_cols = {
    "target_col": dataset_cols["target_col"],
    "series_col": dataset_cols["series_col"],
    "static_cat_cols": [],
    "static_real_cols": [],
    "known_future_real_cols": ["IsHoliday"],
}

HOLIDAY_FEATURE_IDX = 0

print(target_only_dataset_cols)
print("Holiday feature index:", HOLIDAY_FEATURE_IDX)

{'target_col': 'Weekly_Sales_scaled', 'series_col': 'series_id', 'static_cat_cols': [], 'static_real_cols': [], 'known_future_real_cols': ['IsHoliday']}
Holiday feature index: 0


In [38]:
target_only_train_dataset = WalmartPrecomputedTrainingWindowDataset(
    train_panel,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=target_only_dataset_cols["target_col"],
    series_col=target_only_dataset_cols["series_col"],
    static_cat_cols=target_only_dataset_cols["static_cat_cols"],
    static_real_cols=target_only_dataset_cols["static_real_cols"],
    known_future_real_cols=target_only_dataset_cols["known_future_real_cols"],
)

print("Number of training windows:", len(target_only_train_dataset))
print("Available tensor keys:", list(target_only_train_dataset.tensors.keys()))

for key, value in target_only_train_dataset.tensors.items():
    print(key, tuple(value.shape), value.dtype)

Number of training windows: 38464
Available tensor keys: ['past_target', 'future_target', 'past_known_reals', 'future_known_reals', 'static_categoricals', 'static_reals', 'target_mean', 'target_std']
past_target (38464, 52) torch.float32
future_target (38464, 39) torch.float32
past_known_reals (38464, 52, 1) torch.float32
future_known_reals (38464, 39, 1) torch.float32
static_categoricals (38464, 0) torch.int64
static_reals (38464, 0) torch.float32
target_mean (38464,) torch.float32
target_std (38464,) torch.float32


In [39]:
assert len(target_only_train_dataset) > 0

tensors = target_only_train_dataset.tensors

assert tensors["past_target"].shape[1] == CONTEXT_LENGTH
assert tensors["future_target"].shape[1] == PREDICTION_LENGTH

assert tensors["past_known_reals"].shape[1] == CONTEXT_LENGTH
assert tensors["future_known_reals"].shape[1] == PREDICTION_LENGTH

assert tensors["future_known_reals"].shape[2] == 1  # only IsHoliday
assert tensors["past_known_reals"].shape[2] == 1

assert tensors["static_categoricals"].shape[1] == 0
assert tensors["static_reals"].shape[1] == 0

assert not torch.isnan(tensors["past_target"]).any()
assert not torch.isnan(tensors["future_target"]).any()
assert not torch.isnan(tensors["target_mean"]).any()
assert not torch.isnan(tensors["target_std"]).any()

print("Precomputed training dataset checks passed.")

Precomputed training dataset checks passed.


In [40]:
valid_group_sizes = valid_panel.groupby("series_id").size()

full_horizon_series = valid_group_sizes[
    valid_group_sizes == PREDICTION_LENGTH
].index

valid_panel_full = valid_panel[
    valid_panel["series_id"].isin(full_horizon_series)
].copy()

print("Validation series total:", valid_group_sizes.shape[0])
print("Full-horizon validation series:", len(full_horizon_series))

print("valid_panel rows:", len(valid_panel))
print("valid_panel_full rows:", len(valid_panel_full))

assert len(valid_panel_full) == len(full_horizon_series) * PREDICTION_LENGTH

# only for validation, test inference will handle sparse rows by full prediction and merging
print("Full-horizon validation panel prepared.")

Validation series total: 3204
Full-horizon validation series: 2762
valid_panel rows: 115588
valid_panel_full rows: 107718
Full-horizon validation panel prepared.


In [41]:
valid_dataset = WalmartPrecomputedForecastWindowDataset(
    history_df=train_panel,
    future_df=valid_panel_full,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=target_only_dataset_cols["target_col"],
    series_col=target_only_dataset_cols["series_col"],
    static_cat_cols=target_only_dataset_cols["static_cat_cols"],
    static_real_cols=target_only_dataset_cols["static_real_cols"],
    known_future_real_cols=target_only_dataset_cols["known_future_real_cols"],
)

print("Number of validation forecast samples:", len(valid_dataset))
print("Validation tensor keys:", list(valid_dataset.tensors.keys()))

assert len(valid_dataset) == len(full_horizon_series)

for key, value in valid_dataset.tensors.items():
    print(key, tuple(value.shape), value.dtype)

Number of validation forecast samples: 2762
Validation tensor keys: ['past_target', 'future_target', 'past_known_reals', 'future_known_reals', 'static_categoricals', 'static_reals', 'target_mean', 'target_std', 'store', 'dept']
past_target (2762, 52) torch.float32
future_target (2762, 39) torch.float32
past_known_reals (2762, 52, 1) torch.float32
future_known_reals (2762, 39, 1) torch.float32
static_categoricals (2762, 0) torch.int64
static_reals (2762, 0) torch.float32
target_mean (2762,) torch.float32
target_std (2762,) torch.float32
store (2762,) torch.int64
dept (2762,) torch.int64


In [42]:
target_only_train_loader = FastTensorDataLoader(
    target_only_train_dataset.tensors,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

valid_loader = FastTensorDataLoader(
    valid_dataset.tensors,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Train batches per epoch:", len(target_only_train_loader))
print("Validation batches:", len(valid_loader))

Train batches per epoch: 151
Validation batches: 11


In [43]:
target_only_batch  = next(iter(target_only_train_loader))

print("Train batch:")
for key, value in target_only_batch.items():
    print(key, tuple(value.shape), value.dtype)

valid_batch = next(iter(valid_loader))

print("\nValidation batch:")
for key, value in valid_batch.items():
    print(key, tuple(value.shape), value.dtype)

Train batch:
past_target (256, 52) torch.float32
future_target (256, 39) torch.float32
past_known_reals (256, 52, 1) torch.float32
future_known_reals (256, 39, 1) torch.float32
static_categoricals (256, 0) torch.int64
static_reals (256, 0) torch.float32
target_mean (256,) torch.float32
target_std (256,) torch.float32

Validation batch:
past_target (256, 52) torch.float32
future_target (256, 39) torch.float32
past_known_reals (256, 52, 1) torch.float32
future_known_reals (256, 39, 1) torch.float32
static_categoricals (256, 0) torch.int64
static_reals (256, 0) torch.float32
target_mean (256,) torch.float32
target_std (256,) torch.float32
store (256,) torch.int64
dept (256,) torch.int64


In [44]:
sample = valid_dataset[0]

store = int(sample["store"].item())
dept = int(sample["dept"].item())
series_id = f"{store}_{dept}"

history_group = (
    train_panel[train_panel["series_id"] == series_id]
    .sort_values("Date")
    .reset_index(drop=True)
)

expected_past = (
    history_group[dataset_cols["target_col"]]
    .dropna()
    .tail(CONTEXT_LENGTH)
    .to_numpy(dtype=np.float32)
)

actual_past = sample["past_target"].numpy()

if len(expected_past) < CONTEXT_LENGTH:
    expected_past = np.concatenate(
        [
            np.zeros(CONTEXT_LENGTH - len(expected_past), dtype=np.float32),
            expected_past,
        ]
    )

print("series_id:", series_id)
print("history rows:", len(history_group))
print("expected_past shape:", expected_past.shape)
print("actual_past shape:", actual_past.shape)

assert expected_past.shape == actual_past.shape
np.testing.assert_allclose(actual_past, expected_past, rtol=1e-6, atol=1e-6)

print("Validation past_target correctly comes from train history only.")

series_id: 10_1
history rows: 104
expected_past shape: (52,)
actual_past shape: (52,)
Validation past_target correctly comes from train history only.


In [46]:
import time

def time_loader(loader, name="loader", max_batches=None):
    t0 = time.perf_counter()
    n_batches = 0
    n_samples = 0

    for batch in loader:
        n_batches += 1
        n_samples += batch["past_target"].shape[0]

        if max_batches is not None and n_batches >= max_batches:
            break

    elapsed = time.perf_counter() - t0

    print(f"{name}: {elapsed:.2f}s, batches={n_batches}, samples={n_samples}")
    print(f"seconds/batch: {elapsed / max(n_batches, 1):.4f}")

time_loader(target_only_train_loader, "precomputed train_loader full epoch")
time_loader(valid_loader, "valid_loader full eval")

precomputed train_loader full epoch: 0.13s, batches=151, samples=38464
seconds/batch: 0.0009
valid_loader full eval: 0.01s, batches=11, samples=2762
seconds/batch: 0.0009


In [47]:
mlflow.set_experiment("DLinear_Training")

with mlflow.start_run(run_name="DLinear_Preprocessing") as run:
    mlflow.log_param("validation_strategy", "last_39_weeks")
    mlflow.log_param("context_length", CONTEXT_LENGTH)
    mlflow.log_param("prediction_length", PREDICTION_LENGTH)
    mlflow.log_param("base_preprocessor", "WalmartBasePreprocessor")
    mlflow.log_param("neural_preprocessor", "WalmartNeuralPreprocessor")
    mlflow.log_param("target_scaling", "series_mean_std_with_dept_store_global_fallback")
    mlflow.log_param("forecast_validation", "full_horizon_series_only")
    
    mlflow.log_metric("raw_train_rows", len(train))
    mlflow.log_metric("train_split_rows", len(train_raw_part))
    mlflow.log_metric("valid_split_rows", len(valid_raw_part))
    mlflow.log_metric("train_panel_rows", len(train_panel))
    mlflow.log_metric("valid_panel_rows", len(valid_panel))
    mlflow.log_metric("train_windows", len(target_only_train_dataset))
    mlflow.log_metric("valid_forecast_series", len(valid_dataset))
    mlflow.log_metric("num_static_cat_cols", len(dataset_cols["static_cat_cols"]))
    mlflow.log_metric("num_static_real_cols", len(dataset_cols["static_real_cols"]))
    mlflow.log_metric("num_time_varying_features", len(dataset_cols["known_future_real_cols"]))
    
    preprocessing_config = {
        "context_length": CONTEXT_LENGTH,
        "prediction_length": PREDICTION_LENGTH,
        "dataset_cols": dataset_cols,
        "required_base_cols": required_base_cols,
    }
    
    config_path = repo_root / "dlinear_preprocessing_config.json"
    
    with open(config_path, "w") as f:
        json.dump(preprocessing_config, f, indent=2)
    
    mlflow.log_artifact(str(config_path), artifact_path="config")
    config_path.unlink()
    
    preprocessing_run_id = run.info.run_id

print("Logged preprocessing run:", preprocessing_run_id)

🏃 View run DLinear_Preprocessing at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/1/runs/c93af9dc093148bd96fb71345c0165a5
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/1
Logged preprocessing run: c93af9dc093148bd96fb71345c0165a5


### Calendar-aligned validation split

In [81]:
import importlib
import src.data.splits as splits_module

importlib.reload(splits_module)

calendar_aligned_split = splits_module.calendar_aligned_split

In [83]:
calendar_train_raw, calendar_valid_raw = calendar_aligned_split(
    train,
    valid_start="2011-11-04",
    valid_end="2012-07-27",
    date_col="Date",
)

print("Calendar train date range:")
print(calendar_train_raw["Date"].min(), "->", calendar_train_raw["Date"].max())
print("Calendar valid date range:")
print(calendar_valid_raw["Date"].min(), "->", calendar_valid_raw["Date"].max())

print("Unique train dates:", calendar_train_raw["Date"].nunique())
print("Unique validation dates:", calendar_valid_raw["Date"].nunique())

Calendar train date range:
2010-02-05 00:00:00 -> 2011-10-28 00:00:00
Calendar valid date range:
2011-11-04 00:00:00 -> 2012-07-27 00:00:00
Unique train dates: 91
Unique validation dates: 39


In [84]:
calendar_base_preprocessor = WalmartBasePreprocessor()
calendar_base_preprocessor.fit(stores, features)

calendar_train_base = calendar_base_preprocessor.transform(calendar_train_raw)
calendar_valid_base = calendar_base_preprocessor.transform(calendar_valid_raw)

calendar_neural_preprocessor = WalmartNeuralPreprocessor()
calendar_neural_preprocessor.fit(calendar_train_base)

calendar_train_panel = calendar_neural_preprocessor.transform(calendar_train_base)
calendar_valid_panel = calendar_neural_preprocessor.transform(calendar_valid_base)

calendar_dataset_cols = calendar_neural_preprocessor.get_dataset_columns()

In [85]:
calendar_valid_group_sizes = calendar_valid_panel.groupby("series_id").size()

calendar_full_horizon_series = calendar_valid_group_sizes[
    calendar_valid_group_sizes == PREDICTION_LENGTH
].index

calendar_valid_panel_full = calendar_valid_panel[
    calendar_valid_panel["series_id"].isin(calendar_full_horizon_series)
].copy()

print("Calendar validation series total:", calendar_valid_group_sizes.shape[0])
print("Calendar full-horizon validation series:", len(calendar_full_horizon_series))
print("calendar_valid_panel rows:", len(calendar_valid_panel))
print("calendar_valid_panel_full rows:", len(calendar_valid_panel_full))

assert len(calendar_valid_panel_full) == len(calendar_full_horizon_series) * PREDICTION_LENGTH

Calendar validation series total: 3233
Calendar full-horizon validation series: 2762
calendar_valid_panel rows: 115856
calendar_valid_panel_full rows: 107718


In [86]:
calendar_target_only_dataset_cols = {
    "target_col": calendar_dataset_cols["target_col"],
    "series_col": calendar_dataset_cols["series_col"],
    "static_cat_cols": [],
    "static_real_cols": [],
    "known_future_real_cols": ["IsHoliday"],
}

calendar_train_dataset = WalmartPrecomputedTrainingWindowDataset(
    calendar_train_panel,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=calendar_target_only_dataset_cols["target_col"],
    series_col=calendar_target_only_dataset_cols["series_col"],
    static_cat_cols=calendar_target_only_dataset_cols["static_cat_cols"],
    static_real_cols=calendar_target_only_dataset_cols["static_real_cols"],
    known_future_real_cols=calendar_target_only_dataset_cols["known_future_real_cols"],
)

calendar_valid_dataset = WalmartPrecomputedForecastWindowDataset(
    history_df=calendar_train_panel,
    future_df=calendar_valid_panel_full,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=calendar_target_only_dataset_cols["target_col"],
    series_col=calendar_target_only_dataset_cols["series_col"],
    static_cat_cols=calendar_target_only_dataset_cols["static_cat_cols"],
    static_real_cols=calendar_target_only_dataset_cols["static_real_cols"],
    known_future_real_cols=calendar_target_only_dataset_cols["known_future_real_cols"],
)

calendar_train_loader = FastTensorDataLoader(
    calendar_train_dataset.tensors,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

calendar_valid_loader = FastTensorDataLoader(
    calendar_valid_dataset.tensors,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Calendar train windows:", len(calendar_train_dataset))
print("Calendar valid series:", len(calendar_valid_dataset))
print("Calendar train batches:", len(calendar_train_loader))
print("Calendar valid batches:", len(calendar_valid_loader))

Calendar train windows: 2683
Calendar valid series: 2762
Calendar train batches: 11
Calendar valid batches: 11


In [117]:
stable_known_future_cols = [
    col
    for col in calendar_dataset_cols["known_future_real_cols"]
    if "markdown" not in col.lower()
]

print("Original future real features:", len(calendar_dataset_cols["known_future_real_cols"]))
print("Stable future real features:", len(stable_known_future_cols))

print("Removed MarkDown-related features:")
for col in calendar_dataset_cols["known_future_real_cols"]:
    if "markdown" in col.lower():
        print("-", col)

Original future real features: 28
Stable future real features: 11
Removed MarkDown-related features:
- MarkDown1_scaled
- MarkDown2_scaled
- MarkDown3_scaled
- MarkDown4_scaled
- MarkDown5_scaled
- total_markdown_scaled
- abs_total_markdown_scaled
- positive_markdown_sum_scaled
- negative_markdown_sum_scaled
- markdown_missing_count_scaled
- has_markdown_signal
- markdown_available_period
- MarkDown1_was_missing
- MarkDown2_was_missing
- MarkDown3_was_missing
- MarkDown4_was_missing
- MarkDown5_was_missing


In [ ]:
dlinearx_calendar_cols = {
    "target_col": calendar_dataset_cols["target_col"],
    "series_col": calendar_dataset_cols["series_col"],
    "static_cat_cols": calendar_dataset_cols["static_cat_cols"],
    "static_real_cols": calendar_dataset_cols["static_real_cols"],
    "known_future_real_cols": stable_known_future_cols,
}

DLINEARX_HOLIDAY_FEATURE_IDX = dlinearx_calendar_cols["known_future_real_cols"].index("IsHoliday")

TARGET_ONLY_HOLIDAY_FEATURE_IDX = target_only_dataset_cols["known_future_real_cols"].index("IsHoliday")

CALENDAR_TARGET_ONLY_HOLIDAY_FEATURE_IDX = calendar_target_only_dataset_cols["known_future_real_cols"].index("IsHoliday")


In [126]:
dlinearx_train_dataset = WalmartPrecomputedTrainingWindowDataset(
    calendar_train_panel,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=dlinearx_calendar_cols["target_col"],
    series_col=dlinearx_calendar_cols["series_col"],
    static_cat_cols=dlinearx_calendar_cols["static_cat_cols"],
    static_real_cols=dlinearx_calendar_cols["static_real_cols"],
    known_future_real_cols=dlinearx_calendar_cols["known_future_real_cols"],
)

dlinearx_valid_dataset = WalmartPrecomputedForecastWindowDataset(
    history_df=calendar_train_panel,
    future_df=calendar_valid_panel_full,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=dlinearx_calendar_cols["target_col"],
    series_col=dlinearx_calendar_cols["series_col"],
    static_cat_cols=dlinearx_calendar_cols["static_cat_cols"],
    static_real_cols=dlinearx_calendar_cols["static_real_cols"],
    known_future_real_cols=dlinearx_calendar_cols["known_future_real_cols"],
)

dlinearx_train_loader = FastTensorDataLoader(
    dlinearx_train_dataset.tensors,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

dlinearx_valid_loader = FastTensorDataLoader(
    dlinearx_valid_dataset.tensors,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [136]:
stable_known_future_cols = [
    col
    for col in dataset_cols["known_future_real_cols"]
    if "markdown" not in col.lower()
]

full_known_future_cols = dataset_cols["known_future_real_cols"]

dlinearx_last39_stable_cols = {
    "target_col": dataset_cols["target_col"],
    "series_col": dataset_cols["series_col"],
    "static_cat_cols": dataset_cols["static_cat_cols"],
    "static_real_cols": dataset_cols["static_real_cols"],
    "known_future_real_cols": stable_known_future_cols,
}

dlinearx_last39_full_cols = {
    "target_col": dataset_cols["target_col"],
    "series_col": dataset_cols["series_col"],
    "static_cat_cols": dataset_cols["static_cat_cols"],
    "static_real_cols": dataset_cols["static_real_cols"],
    "known_future_real_cols": full_known_future_cols,
}

In [137]:
def build_dlinearx_loaders(
    train_panel,
    valid_panel_full,
    cols,
    batch_size=512,
):
    train_dataset = WalmartPrecomputedTrainingWindowDataset(
        train_panel,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        target_col=cols["target_col"],
        series_col=cols["series_col"],
        static_cat_cols=cols["static_cat_cols"],
        static_real_cols=cols["static_real_cols"],
        known_future_real_cols=cols["known_future_real_cols"],
    )

    valid_dataset = WalmartPrecomputedForecastWindowDataset(
        history_df=train_panel,
        future_df=valid_panel_full,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        target_col=cols["target_col"],
        series_col=cols["series_col"],
        static_cat_cols=cols["static_cat_cols"],
        static_real_cols=cols["static_real_cols"],
        known_future_real_cols=cols["known_future_real_cols"],
    )

    train_loader = FastTensorDataLoader(
        train_dataset.tensors,
        batch_size=batch_size,
        shuffle=True,
    )

    valid_loader = FastTensorDataLoader(
        valid_dataset.tensors,
        batch_size=batch_size,
        shuffle=False,
    )

    static_cat_cardinalities = [
        int(max(train_panel[col].max(), valid_panel_full[col].max())) + 1
        for col in cols["static_cat_cols"]
    ]

    num_future_reals = len(cols["known_future_real_cols"])
    num_static_reals = len(cols["static_real_cols"])
    holiday_feature_idx = cols["known_future_real_cols"].index("IsHoliday")

    assert train_dataset.tensors["future_known_reals"].shape[2] == num_future_reals
    assert valid_dataset.tensors["future_known_reals"].shape[2] == num_future_reals

    return {
        "train_dataset": train_dataset,
        "valid_dataset": valid_dataset,
        "train_loader": train_loader,
        "valid_loader": valid_loader,
        "static_cat_cardinalities": static_cat_cardinalities,
        "num_future_reals": num_future_reals,
        "num_static_reals": num_static_reals,
        "holiday_feature_idx": holiday_feature_idx,
    }

In [140]:
last39_stable_data = build_dlinearx_loaders(
    train_panel=train_panel,
    valid_panel_full=valid_panel_full,
    cols=dlinearx_last39_stable_cols,
    batch_size=BATCH_SIZE,
)

last39_full_data = build_dlinearx_loaders(
    train_panel=train_panel,
    valid_panel_full=valid_panel_full,
    cols=dlinearx_last39_full_cols,
    batch_size=BATCH_SIZE,
)

In [141]:
print("Stable future features:", last39_stable_data["num_future_reals"])
print("Full future features:", last39_full_data["num_future_reals"])

print("Removed features:")
for col in full_known_future_cols:
    if col not in stable_known_future_cols:
        print("-", col)

Stable future features: 11
Full future features: 28
Removed features:
- MarkDown1_scaled
- MarkDown2_scaled
- MarkDown3_scaled
- MarkDown4_scaled
- MarkDown5_scaled
- total_markdown_scaled
- abs_total_markdown_scaled
- positive_markdown_sum_scaled
- negative_markdown_sum_scaled
- markdown_missing_count_scaled
- has_markdown_signal
- markdown_available_period
- MarkDown1_was_missing
- MarkDown2_was_missing
- MarkDown3_was_missing
- MarkDown4_was_missing
- MarkDown5_was_missing


In [145]:
removed_markdown_cols = [
    col
    for col in dataset_cols["known_future_real_cols"]
    if "markdown" in col.lower()
]

print("Removed MarkDown-related features:", len(removed_markdown_cols))
for col in removed_markdown_cols:
    print("-", col)

Removed MarkDown-related features: 17
- MarkDown1_scaled
- MarkDown2_scaled
- MarkDown3_scaled
- MarkDown4_scaled
- MarkDown5_scaled
- total_markdown_scaled
- abs_total_markdown_scaled
- positive_markdown_sum_scaled
- negative_markdown_sum_scaled
- markdown_missing_count_scaled
- has_markdown_signal
- markdown_available_period
- MarkDown1_was_missing
- MarkDown2_was_missing
- MarkDown3_was_missing
- MarkDown4_was_missing
- MarkDown5_was_missing


In [146]:
safe_markdown_cols = {
    "has_markdown_signal",
    "markdown_available_period",
    "markdown_missing_count",
    "MarkDown1_was_missing",
    "MarkDown2_was_missing",
    "MarkDown3_was_missing",
    "MarkDown4_was_missing",
    "MarkDown5_was_missing",
}

stable_cols_set = set(stable_known_future_cols)

markdown_flags_only_cols = [
    col
    for col in dataset_cols["known_future_real_cols"]
    if col in stable_cols_set or col in safe_markdown_cols
]

print("Stable no-MarkDown feature count:", len(stable_known_future_cols))
print("MarkDown-flags-only feature count:", len(markdown_flags_only_cols))

print("Added bounded MarkDown features:")
for col in markdown_flags_only_cols:
    if col not in stable_cols_set:
        print("-", col)

Stable no-MarkDown feature count: 11
MarkDown-flags-only feature count: 18
Added bounded MarkDown features:
- has_markdown_signal
- markdown_available_period
- MarkDown1_was_missing
- MarkDown2_was_missing
- MarkDown3_was_missing
- MarkDown4_was_missing
- MarkDown5_was_missing


In [147]:
dlinearx_last39_flags_cols = {
    "target_col": dataset_cols["target_col"],
    "series_col": dataset_cols["series_col"],
    "static_cat_cols": dataset_cols["static_cat_cols"],
    "static_real_cols": dataset_cols["static_real_cols"],
    "known_future_real_cols": markdown_flags_only_cols,
}

In [149]:
last39_flags_data = build_dlinearx_loaders(
    train_panel=train_panel,
    valid_panel_full=valid_panel_full,
    cols=dlinearx_last39_flags_cols,
    batch_size=BATCH_SIZE,
)

## Evaluation utilities

In [30]:
HOLIDAY_FEATURE_IDX = dataset_cols["known_future_real_cols"].index("IsHoliday")
print("IsHoliday index in future_known_reals:", HOLIDAY_FEATURE_IDX)


def weighted_mae_np(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    is_holiday = np.asarray(is_holiday).reshape(-1).astype(bool)
    
    weights = np.where(is_holiday, 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def inverse_scale_torch(y_scaled, target_mean, target_std):
    """
    y_scaled: [B, H]
    target_mean: [B]
    target_std: [B]
    """
    return y_scaled * target_std.unsqueeze(1) + target_mean.unsqueeze(1)


def move_batch_to_device(batch, device):
    return {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

IsHoliday index in future_known_reals: 16


## DLinear model definitions

In [31]:
class MovingAverage(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        
        if kernel_size % 2 == 0:
            raise ValueError("kernel_size should be odd so the output length matches the input length.")
        
        self.kernel_size = kernel_size
        self.pad = (kernel_size - 1) // 2
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)
    
    def forward(self, x):
        """
        x: [B, L]
        """
        front = x[:, :1].repeat(1, self.pad)
        end = x[:, -1:].repeat(1, self.pad)
        x_padded = torch.cat([front, x, end], dim=1)
        
        trend = self.avg(x_padded.unsqueeze(1)).squeeze(1)
        return trend


class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.moving_average = MovingAverage(kernel_size)
    
    def forward(self, x):
        """
        x: [B, L]
        returns:
            seasonal/residual: [B, L]
            trend: [B, L]
        """
        trend = self.moving_average(x)
        seasonal = x - trend
        return seasonal, trend


class DLinear(nn.Module):
    def __init__(
        self,
        context_length: int,
        prediction_length: int,
        kernel_size: int = 25,
    ):
        super().__init__()
        
        self.context_length = context_length
        self.prediction_length = prediction_length
        self.kernel_size = kernel_size
        
        self.decomposition = SeriesDecomposition(kernel_size)
        self.seasonal_linear = nn.Linear(context_length, prediction_length)
        self.trend_linear = nn.Linear(context_length, prediction_length)
    
    def forward(self, past_target):
        """
        past_target: [B, context_length]
        returns scaled forecast: [B, prediction_length]
        """
        seasonal, trend = self.decomposition(past_target)
        
        seasonal_forecast = self.seasonal_linear(seasonal)
        trend_forecast = self.trend_linear(trend)
        
        return seasonal_forecast + trend_forecast

In [ ]:
class DLinearX(nn.Module):
    def __init__(
        self,
        context_length: int,
        prediction_length: int,
        static_cat_cardinalities,
        num_future_reals: int,
        num_static_reals: int,
        kernel_size: int = 25,
        embedding_dim: int = 8,
        hidden_dim: int = 64,
        dropout: float = 0.1,
    ):
        super().__init__()
        
        self.context_length = context_length
        self.prediction_length = prediction_length
        self.kernel_size = kernel_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        
        self.backbone = DLinear(
            context_length=context_length,
            prediction_length=prediction_length,
            kernel_size=kernel_size,
        )
        
        # embeddings instead of OHE
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=cardinality, embedding_dim=embedding_dim)
            for cardinality in static_cat_cardinalities
        ])
        
        static_embedding_dim = len(static_cat_cardinalities) * embedding_dim
        correction_input_dim = (
            num_future_reals
            + static_embedding_dim
            + num_static_reals
        )
        
        self.correction_head = nn.Sequential(
            nn.Linear(correction_input_dim, hidden_dim),
            nn.ReLU(), # pinch of nonlinearity
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(
        self,
        past_target,
        future_known_reals,
        static_categoricals,
        static_reals,
    ):
        """
        past_target: [B, context_length]
        future_known_reals: [B, prediction_length, F]
        static_categoricals: [B, C]
        static_reals: [B, S]
        """
        base_forecast = self.backbone(past_target)
        
        batch_size, horizon, _ = future_known_reals.shape
        
        if len(self.embeddings) > 0:
            embedded_parts = [
                embedding(static_categoricals[:, i])
                for i, embedding in enumerate(self.embeddings)
            ]
            static_embedded = torch.cat(embedded_parts, dim=1)
        else:
            static_embedded = torch.empty(
                batch_size,
                0,
                device=future_known_reals.device,
            )
        
        static_context = torch.cat([static_embedded, static_reals], dim=1)
        static_context = static_context.unsqueeze(1).expand(-1, horizon, -1)
        
        correction_input = torch.cat(
            [future_known_reals, static_context],
            dim=-1,
        )
        
        correction = self.correction_head(correction_input).squeeze(-1)
        
        return base_forecast + correction

In [121]:
batch = next(iter(dlinearx_train_loader))

test_batch = move_batch_to_device(batch, DEVICE)

target_only_model = DLinear(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    kernel_size=25,
).to(DEVICE)

with torch.no_grad():
    out = target_only_model(test_batch["past_target"])

print("Target-only output shape:", tuple(out.shape))
assert out.shape == test_batch["future_target"].shape

del target_only_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("DLinear target-only shape test passed.")

Target-only output shape: (256, 39)
DLinear target-only shape test passed.


In [144]:
static_cat_cardinalities = [
    int(max(
        calendar_train_panel[col].max(),
        calendar_valid_panel_full[col].max(),
    )) + 1
    for col in dlinearx_calendar_cols["static_cat_cols"]
]

num_future_reals = len(dlinearx_calendar_cols["known_future_real_cols"])
num_static_reals = len(dlinearx_calendar_cols["static_real_cols"])

print("Static categorical cardinalities:", static_cat_cardinalities)
print("Number of future real features:", num_future_reals)
print("Number of static real features:", num_static_reals)

dlinearx_model = DLinearX(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    static_cat_cardinalities=static_cat_cardinalities,
    num_future_reals=num_future_reals,
    num_static_reals=num_static_reals,
    kernel_size=25,
    embedding_dim=8,
    hidden_dim=64,
    dropout=0.1,
).to(DEVICE)

with torch.no_grad():
    out = dlinearx_model(
        past_target=test_batch["past_target"],
        future_known_reals=test_batch["future_known_reals"],
        static_categoricals=test_batch["static_categoricals"],
        static_reals=test_batch["static_reals"],
    )

print("DLinear-X output shape:", tuple(out.shape))
assert out.shape == test_batch["future_target"].shape

del dlinearx_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("DLinear-X shape test passed.")

Static categorical cardinalities: [46, 82, 4]
Number of future real features: 11
Number of static real features: 1
DLinear-X output shape: (256, 39)
DLinear-X shape test passed.


## Training utilities

In [50]:
def forward_model(model, batch, model_variant: str):
    if model_variant == "target_only":
        return model(batch["past_target"])
    
    if model_variant == "dlinearx":
        return model(
            past_target=batch["past_target"],
            future_known_reals=batch["future_known_reals"],
            static_categoricals=batch["static_categoricals"],
            static_reals=batch["static_reals"],
        )
    
    raise ValueError(f"Unknown model_variant: {model_variant}")


def train_one_epoch(model, loader, optimizer, loss_fn, device, model_variant: str):
    model.train()
    
    total_loss = 0.0
    total_items = 0
    
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        
        optimizer.zero_grad(set_to_none=True)
        
        preds_scaled = forward_model(model, batch, model_variant)
        target_scaled = batch["future_target"]
        
        loss = loss_fn(preds_scaled, target_scaled)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        n_items = target_scaled.numel()
        total_loss += loss.item() * n_items
        total_items += n_items
    
    return total_loss / total_items

In [51]:
@torch.no_grad()
def evaluate_model(model, loader, loss_fn, device, model_variant: str, holiday_feature_idx: int):
    model.eval()
    
    total_loss = 0.0
    total_items = 0
    
    all_true = []
    all_pred = []
    all_holiday = []
    
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        
        preds_scaled = forward_model(model, batch, model_variant)
        target_scaled = batch["future_target"]
        
        loss = loss_fn(preds_scaled, target_scaled)
        
        n_items = target_scaled.numel()
        total_loss += loss.item() * n_items
        total_items += n_items
        
        preds_original = inverse_scale_torch(
            preds_scaled,
            batch["target_mean"],
            batch["target_std"],
        )
        
        target_original = inverse_scale_torch(
            target_scaled,
            batch["target_mean"],
            batch["target_std"],
        )
        
        is_holiday = batch["future_known_reals"][:, :, holiday_feature_idx]
        
        all_pred.append(preds_original.detach().cpu().numpy())
        all_true.append(target_original.detach().cpu().numpy())
        all_holiday.append(is_holiday.detach().cpu().numpy())
    
    y_pred = np.concatenate(all_pred, axis=0)
    y_true = np.concatenate(all_true, axis=0)
    is_holiday = np.concatenate(all_holiday, axis=0)
    
    valid_loss = total_loss / total_items
    valid_wmae = weighted_mae_np(y_true, y_pred, is_holiday)
    valid_mae = np.mean(np.abs(y_true.reshape(-1) - y_pred.reshape(-1)))
    
    return {
        "valid_loss": valid_loss,
        "valid_wmae": valid_wmae,
        "valid_mae": valid_mae,
    }

In [52]:
def fit_model(
    model,
    train_loader,
    valid_loader,
    optimizer,
    loss_fn,
    device,
    model_variant: str,
    holiday_feature_idx: int,
    epochs: int,
):
    best_state_dict = None
    best_valid_wmae = float("inf")
    best_epoch = None
    
    history = []
    
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
            model_variant=model_variant,
        )
        
        valid_metrics = evaluate_model(
            model=model,
            loader=valid_loader,
            loss_fn=loss_fn,
            device=device,
            model_variant=model_variant,
            holiday_feature_idx=holiday_feature_idx,
        )
        
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **valid_metrics,
        }
        history.append(row)
        
        if mlflow.active_run() is not None:
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("valid_loss", valid_metrics["valid_loss"], step=epoch)
            mlflow.log_metric("valid_wmae", valid_metrics["valid_wmae"], step=epoch)
            mlflow.log_metric("valid_mae", valid_metrics["valid_mae"], step=epoch)
        
        if valid_metrics["valid_wmae"] < best_valid_wmae:
            best_valid_wmae = valid_metrics["valid_wmae"]
            best_epoch = epoch
            best_state_dict = deepcopy(model.state_dict())
        
        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.5f} | "
            f"valid_loss={valid_metrics['valid_loss']:.5f} | "
            f"valid_wmae={valid_metrics['valid_wmae']:.2f} | "
            f"valid_mae={valid_metrics['valid_mae']:.2f}"
        )
    
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    
    history_df = pd.DataFrame(history)
    
    return {
        "model": model,
        "history": history_df,
        "best_valid_wmae": best_valid_wmae,
        "best_epoch": best_epoch,
        "best_state_dict": best_state_dict,
    }

In [53]:
@torch.no_grad()
def collect_validation_predictions(
    model,
    loader,
    dataset,
    device,
    model_variant: str,
    holiday_feature_idx: int,
):
    model.eval()
    
    all_true = []
    all_pred = []
    all_holiday = []
    
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        
        preds_scaled = forward_model(model, batch, model_variant)
        target_scaled = batch["future_target"]
        
        preds_original = inverse_scale_torch(
            preds_scaled,
            batch["target_mean"],
            batch["target_std"],
        )
        
        target_original = inverse_scale_torch(
            target_scaled,
            batch["target_mean"],
            batch["target_std"],
        )
        
        is_holiday = batch["future_known_reals"][:, :, holiday_feature_idx]
        
        all_pred.append(preds_original.detach().cpu().numpy())
        all_true.append(target_original.detach().cpu().numpy())
        all_holiday.append(is_holiday.detach().cpu().numpy())
    
    y_pred = np.concatenate(all_pred, axis=0).reshape(-1)
    y_true = np.concatenate(all_true, axis=0).reshape(-1)
    is_holiday = np.concatenate(all_holiday, axis=0).reshape(-1).astype(bool)
    
    index_df = dataset.get_future_index().reset_index(drop=True).copy()
    
    assert len(index_df) == len(y_pred)
    
    index_df["y_true"] = y_true
    index_df["y_pred"] = y_pred
    index_df["abs_error"] = np.abs(index_df["y_true"] - index_df["y_pred"])
    index_df["IsHoliday_eval"] = is_holiday
    
    return index_df

In [54]:
def plot_loss_history(history_df, title, output_path):
    fig = plt.figure()
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["valid_loss"], label="valid_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Scaled loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def plot_wmae_history(history_df, title, output_path):
    fig = plt.figure()
    plt.plot(history_df["epoch"], history_df["valid_wmae"], label="valid_wmae")
    plt.xlabel("Epoch")
    plt.ylabel("Validation WMAE")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

In [61]:
def make_loss_fn(loss_name: str):
    loss_name = loss_name.lower()

    if loss_name == "mse":
        return torch.nn.MSELoss()

    if loss_name == "mae" or loss_name == "l1":
        return torch.nn.L1Loss()

    if loss_name == "huber":
        return torch.nn.HuberLoss(delta=1.0)

    raise ValueError(f"Unknown loss_name: {loss_name}")

In [55]:
ARTIFACT_DIR = repo_root / "artifacts" / "dlinear"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

experiment_results = []

print("Artifact dir:", ARTIFACT_DIR)

Artifact dir: C:\Users\Myvari\Desktop\walmart-sales-forecasting\artifacts\dlinear


## Target-only DLinear

In [62]:
# TARGET_ONLY_CONFIG = {
#     "model_type": "DLinear",
#     "model_variant": "target_only",
#     "feature_set": "past_target_only",
#     "context_length": CONTEXT_LENGTH,
#     "prediction_length": PREDICTION_LENGTH,
#     "kernel_size": 25,
#     "batch_size": BATCH_SIZE,
#     "learning_rate": 1e-3,
#     "weight_decay": 1e-4,
#     "epochs": 10,
#     "loss_function": "MSELoss",
#     "optimizer": "AdamW",
# }

DLINEAR_TARGET_ONLY_GRID = [
    {"kernel_size": 7,  "lr": 1e-3, "weight_decay": 0.0,   "loss_name": "mse"},
    {"kernel_size": 13, "lr": 1e-3, "weight_decay": 0.0,   "loss_name": "mse"},
    {"kernel_size": 25, "lr": 1e-3, "weight_decay": 0.0,   "loss_name": "mse"},
    {"kernel_size": 39, "lr": 1e-3, "weight_decay": 0.0,   "loss_name": "mse"},
    {"kernel_size": 51, "lr": 1e-3, "weight_decay": 0.0,   "loss_name": "mse"},

    {"kernel_size": 13, "lr": 5e-4, "weight_decay": 1e-4, "loss_name": "mse"},
    {"kernel_size": 25, "lr": 5e-4, "weight_decay": 1e-4, "loss_name": "mse"},
    {"kernel_size": 39, "lr": 5e-4, "weight_decay": 1e-4, "loss_name": "mse"},

    {"kernel_size": 13, "lr": 5e-4, "weight_decay": 1e-4, "loss_name": "huber"},
    {"kernel_size": 25, "lr": 5e-4, "weight_decay": 1e-4, "loss_name": "huber"},
    {"kernel_size": 39, "lr": 5e-4, "weight_decay": 1e-4, "loss_name": "huber"},
]

set_seed(SEED)

# target_only_model = DLinear(
#     context_length=CONTEXT_LENGTH,
#     prediction_length=PREDICTION_LENGTH,
#     kernel_size=TARGET_ONLY_CONFIG["kernel_size"],
# ).to(DEVICE)

# print(target_only_model)
# print("Trainable parameters:", count_parameters(target_only_model))

In [63]:
len(target_only_train_loader)

151

In [ ]:
from copy import deepcopy
import pandas as pd
import mlflow
import torch

dlinear_target_only_results = []

EPOCHS_SEARCH = 10

for i, config in enumerate(DLINEAR_TARGET_ONLY_GRID, start=1):
    run_name = (
        f"DLinear_TargetOnly_"
        f"k{config['kernel_size']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"{config['loss_name']}"
    )

    print("=" * 80)
    print(f"Run {i}/{len(DLINEAR_TARGET_ONLY_GRID)}: {run_name}")
    print(config)

    model = DLinear(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        kernel_size=config["kernel_size"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "DLinear",
            "model_variant": "target_only",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "kernel_size": config["kernel_size"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_SEARCH,
            "validation_strategy": "last_39_weeks",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(target_only_train_dataset),
            "valid_series": len(valid_dataset),
        })

        result = fit_model(
            model=model,
            train_loader=target_only_train_loader,
            valid_loader=valid_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            model_variant="target_only",
            holiday_feature_idx=TARGET_ONLY_HOLIDAY_FEATURE_IDX,
            epochs=EPOCHS_SEARCH,
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("best_valid_wmae", best_wmae)
        mlflow.log_metric("best_epoch", best_epoch)

        dlinear_target_only_results.append({
            "run_name": run_name,
            "run_id": run_id,
            **config,
            "best_valid_wmae": best_wmae,
            "best_epoch": best_epoch,
        })

        print(f"Best WMAE: {best_wmae:.2f}")
        print(f"Best epoch: {best_epoch}")

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

dlinear_target_only_results_df = (
    pd.DataFrame(dlinear_target_only_results)
    .sort_values("best_valid_wmae")
    .reset_index(drop=True)
)

display(dlinear_target_only_results_df)

Run 1/11: DLinear_TargetOnly_k7_lr0.001_wd0.0_mse
{'kernel_size': 7, 'lr': 0.001, 'weight_decay': 0.0, 'loss_name': 'mse'}
Epoch 001 | train_loss=0.63070 | valid_loss=1.23117 | valid_wmae=2626.07 | valid_mae=2626.66
Epoch 002 | train_loss=0.42460 | valid_loss=1.23370 | valid_wmae=2614.12 | valid_mae=2609.45
Epoch 003 | train_loss=0.40153 | valid_loss=1.22623 | valid_wmae=2589.26 | valid_mae=2588.88
Epoch 004 | train_loss=0.39508 | valid_loss=1.22330 | valid_wmae=2581.23 | valid_mae=2575.66
Epoch 005 | train_loss=0.39302 | valid_loss=1.22808 | valid_wmae=2593.34 | valid_mae=2589.29
Epoch 006 | train_loss=0.39227 | valid_loss=1.22256 | valid_wmae=2588.46 | valid_mae=2578.71
Epoch 007 | train_loss=0.39204 | valid_loss=1.22486 | valid_wmae=2586.15 | valid_mae=2580.27
Epoch 008 | train_loss=0.39196 | valid_loss=1.22748 | valid_wmae=2586.66 | valid_mae=2578.63
Epoch 009 | train_loss=0.39197 | valid_loss=1.22399 | valid_wmae=2584.26 | valid_mae=2578.36
Epoch 010 | train_loss=0.39199 | valid_l

,run_name,run_id,kernel_size,lr,weight_decay,loss_name,best_valid_wmae,best_epoch
0,DLinear_TargetOnly_k13_lr0.0005_wd0.0001_huber,4d36f4e63808492d80689313dc6b3f7a,13,0.0005,0.0001,huber,2564.486226,7
1,DLinear_TargetOnly_k39_lr0.0005_wd0.0001_huber,2458018b0300446cac57fd80c6d7ebfe,39,0.0005,0.0001,huber,2565.108116,6
2,DLinear_TargetOnly_k25_lr0.0005_wd0.0001_huber,d358d549e48d42a2a088e2f7a469d21d,25,0.0005,0.0001,huber,2566.427561,9
3,DLinear_TargetOnly_k39_lr0.001_wd0.0_mse,7c12f9e13b144188b27fbbf4a88e2236,39,0.0010,0.0000,mse,2577.954261,9
4,DLinear_TargetOnly_k25_lr0.001_wd0.0_mse,f0655311057344249e0c6656f9dcae2d,25,0.0010,0.0000,mse,2580.270023,10
5,DLinear_TargetOnly_k7_lr0.001_wd0.0_mse,cc2cd02a3a524eb8ae17c09e927d4c5e,7,0.0010,0.0000,mse,2581.234418,4
6,DLinear_TargetOnly_k51_lr0.001_wd0.0_mse,7d881aeee0c74c27bff76ce42a00faba,51,0.0010,0.0000,mse,2583.602305,10
7,DLinear_TargetOnly_k13_lr0.0005_wd0.0001_mse,760193b0b31844e8aba9b9da5f0b5d9c,13,0.0005,0.0001,mse,2584.613834,7
8,DLinear_TargetOnly_k13_lr0.001_wd0.0_mse,1c3771cfcb7940799aa6a440170a0512,13,0.0010,0.0000,mse,2585.452825,9
9,DLinear_TargetOnly_k39_lr0.0005_wd0.0001_mse,bf37150de37640b097d937e2652f070a,39,0.0005,0.0001,mse,2589.464090,9


In [88]:
top_configs = (
    dlinear_target_only_results_df
    .sort_values("best_valid_wmae")
    .head(6)
    .copy()
)

In [89]:
diverse_configs = []

# best small kernel
diverse_configs.append(
    dlinear_target_only_results_df[
        dlinear_target_only_results_df["kernel_size"].isin([7, 13])
    ].sort_values("best_valid_wmae").head(1)
)

# best large kernel
diverse_configs.append(
    dlinear_target_only_results_df[
        dlinear_target_only_results_df["kernel_size"].isin([39, 51])
    ].sort_values("best_valid_wmae").head(1)
)

# best huber
diverse_configs.append(
    dlinear_target_only_results_df[
        dlinear_target_only_results_df["loss_name"] == "huber"
    ].sort_values("best_valid_wmae").head(1)
)

calendar_candidate_configs = pd.concat(
    [top_configs] + diverse_configs,
    axis=0,
).drop_duplicates(
    subset=["kernel_size", "lr", "weight_decay", "loss_name"]
).reset_index(drop=True)

display(calendar_candidate_configs)

,run_name,run_id,kernel_size,lr,weight_decay,loss_name,best_valid_wmae,best_epoch
0,DLinear_TargetOnly_k13_lr0.0005_wd0.0001_huber,4d36f4e63808492d80689313dc6b3f7a,13,0.0005,0.0001,huber,2564.486226,7
1,DLinear_TargetOnly_k39_lr0.0005_wd0.0001_huber,2458018b0300446cac57fd80c6d7ebfe,39,0.0005,0.0001,huber,2565.108116,6
2,DLinear_TargetOnly_k25_lr0.0005_wd0.0001_huber,d358d549e48d42a2a088e2f7a469d21d,25,0.0005,0.0001,huber,2566.427561,9
3,DLinear_TargetOnly_k39_lr0.001_wd0.0_mse,7c12f9e13b144188b27fbbf4a88e2236,39,0.0010,0.0000,mse,2577.954261,9
4,DLinear_TargetOnly_k25_lr0.001_wd0.0_mse,f0655311057344249e0c6656f9dcae2d,25,0.0010,0.0000,mse,2580.270023,10
5,DLinear_TargetOnly_k7_lr0.001_wd0.0_mse,cc2cd02a3a524eb8ae17c09e927d4c5e,7,0.0010,0.0000,mse,2581.234418,4


In [ ]:
calendar_results = []


extra_configs_v1 = [
    # best current setup but faster learning
    {"kernel_size": 13, "lr": 1e-3, "weight_decay": 1e-4, "loss_name": "huber"},
    {"kernel_size": 13, "lr": 2e-3, "weight_decay": 1e-4, "loss_name": "huber"},

    # 5e-3 may be too aggressive, but cheap to test
    {"kernel_size": 13, "lr": 5e-3, "weight_decay": 1e-4, "loss_name": "huber"},

    # small kernel was close with MSE, so test it with Huber too
    {"kernel_size": 7, "lr": 1e-3, "weight_decay": 1e-4, "loss_name": "huber"},
    {"kernel_size": 7, "lr": 2e-3, "weight_decay": 1e-4, "loss_name": "huber"},

    # current second-best family, maybe higher LR improves it
    {"kernel_size": 7, "lr": 2e-3, "weight_decay": 0.0, "loss_name": "mse"},
]

EPOCHS_CALENDAR = 25

# for i, row in calendar_candidate_configs.iterrows():
for i, config in enumerate(extra_configs_v1):
    config = {
        "kernel_size": int(config["kernel_size"]),
        "lr": float(config["lr"]),
        "weight_decay": float(config["weight_decay"]),
        "loss_name": config["loss_name"],
    }

    run_name = (
        f"DLinear_TargetOnly_CalendarAligned_"
        f"k{config['kernel_size']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"{config['loss_name']}"
    )

    print("=" * 80)
    # print(f"Calendar run {i+1}/{len(calendar_candidate_configs)}: {run_name}")
    print(f"Calendar run {i+1}/{len(extra_configs_v1)}: {run_name}")
    print(config)

    model = DLinear(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        kernel_size=config["kernel_size"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "DLinear",
            "model_variant": "target_only",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "kernel_size": config["kernel_size"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_CALENDAR,
            "validation_strategy": "calendar_aligned_39_weeks",
            "calendar_valid_start": "2011-11-04",
            "calendar_valid_end": "2012-07-27",
            "train_windows": len(calendar_train_dataset),
            "valid_series": len(calendar_valid_dataset),
        })

        result = fit_model(
            model=model,
            train_loader=calendar_train_loader,
            valid_loader=calendar_valid_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            model_variant="target_only",
            holiday_feature_idx=CALENDAR_TARGET_ONLY_HOLIDAY_FEATURE_IDX,
            epochs=EPOCHS_CALENDAR,
        )

        calendar_results.append({
            **config,
            "run_name": run_name,
            "run_id": run_id,
            "calendar_best_valid_wmae": result["best_valid_wmae"],
            "calendar_best_epoch": result["best_epoch"],
        })

        mlflow.log_metric("calendar_best_valid_wmae", result["best_valid_wmae"])
        mlflow.log_metric("calendar_best_epoch", result["best_epoch"])

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

calendar_results_df = (
    pd.DataFrame(calendar_results)
    .sort_values("calendar_best_valid_wmae")
    .reset_index(drop=True)
)

display(calendar_results_df)

Calendar run 1/6: DLinear_TargetOnly_CalendarAligned_k13_lr0.001_wd0.0001_huber
{'kernel_size': 13, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'huber'}
Epoch 001 | train_loss=0.39351 | valid_loss=0.63270 | valid_wmae=3955.41 | valid_mae=3683.68
Epoch 002 | train_loss=0.31872 | valid_loss=0.61431 | valid_wmae=3841.84 | valid_mae=3562.40
Epoch 003 | train_loss=0.27205 | valid_loss=0.60257 | valid_wmae=3756.24 | valid_mae=3470.17
Epoch 004 | train_loss=0.24163 | valid_loss=0.59584 | valid_wmae=3695.04 | valid_mae=3403.80
Epoch 005 | train_loss=0.22042 | valid_loss=0.59284 | valid_wmae=3659.53 | valid_mae=3362.80
Epoch 006 | train_loss=0.20538 | valid_loss=0.59059 | valid_wmae=3633.39 | valid_mae=3329.88
Epoch 007 | train_loss=0.19456 | valid_loss=0.58824 | valid_wmae=3607.08 | valid_mae=3298.69
Epoch 008 | train_loss=0.18642 | valid_loss=0.58561 | valid_wmae=3585.72 | valid_mae=3272.63
Epoch 009 | train_loss=0.18020 | valid_loss=0.58376 | valid_wmae=3566.55 | valid_mae=3250.45
Epoc

,kernel_size,lr,weight_decay,loss_name,run_name,run_id,calendar_best_valid_wmae,calendar_best_epoch
0,13,0.001,0.0001,huber,DLinear_TargetOnly_CalendarAligned_k13_lr0.001...,c42bc2650b474d0a88a67f8b84e6c3bd,3461.694668,24
1,13,0.005,0.0001,huber,DLinear_TargetOnly_CalendarAligned_k13_lr0.005...,a18c1d11493a443ebb568d05c3e9bf5b,3474.052039,6
2,13,0.002,0.0001,huber,DLinear_TargetOnly_CalendarAligned_k13_lr0.002...,cf2bc1f5a9c94d6393056240f7ee8ab9,3489.559162,25
3,7,0.002,0.0000,mse,DLinear_TargetOnly_CalendarAligned_k7_lr0.002_...,a42d51a3805e474a88994706e841728b,3496.491686,22
4,7,0.002,0.0001,huber,DLinear_TargetOnly_CalendarAligned_k7_lr0.002_...,445f16d644fd4b39a67e674e5266abdf,3497.508134,21
5,7,0.001,0.0001,huber,DLinear_TargetOnly_CalendarAligned_k7_lr0.001_...,fbc4e81781f047cf8b5acfd01b6aa2f0,3498.443935,25


## Compare split accuracy for validation

In [ ]:

extra_configs_v2 = [
    {"kernel_size": 13, "lr": 1e-3, "weight_decay": 1e-4, "loss_name": "huber"},
]


dlinear_target_only_results1 = []

EPOCHS_SEARCH1 = 25

for i, config in enumerate(extra_configs_v2, start=1):
    run_name = (
        f"DLinear_TargetOnly_"
        f"k{config['kernel_size']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"{config['loss_name']}"
    )

    print("=" * 80)
    print(f"Run {i}/{len(extra_configs_v2)}: {run_name}")
    print(config)

    model = DLinear(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        kernel_size=config["kernel_size"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "DLinear",
            "model_variant": "target_only",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "kernel_size": config["kernel_size"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_SEARCH1,
            "validation_strategy": "last_39_weeks",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(target_only_train_dataset),
            "valid_series": len(valid_dataset),
        })

        result = fit_model(
            model=model,
            train_loader=target_only_train_loader,
            valid_loader=valid_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            model_variant="target_only",
            holiday_feature_idx=TARGET_ONLY_HOLIDAY_FEATURE_IDX,
            epochs=EPOCHS_SEARCH1,
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("best_valid_wmae", best_wmae)
        mlflow.log_metric("best_epoch", best_epoch)

        dlinear_target_only_results1.append({
            "run_name": run_name,
            "run_id": run_id,
            **config,
            "best_valid_wmae": best_wmae,
            "best_epoch": best_epoch,
        })

        print(f"Best WMAE: {best_wmae:.2f}")
        print(f"Best epoch: {best_epoch}")

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

dlinear_target_only_results_df1 = (
    pd.DataFrame(dlinear_target_only_results1)
    .sort_values("best_valid_wmae")
    .reset_index(drop=True)
)

display(dlinear_target_only_results_df1)

Run 1/1: DLinear_TargetOnly_k13_lr0.001_wd0.0001_huber
{'kernel_size': 13, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'huber'}
Epoch 001 | train_loss=0.25855 | valid_loss=0.44734 | valid_wmae=2628.75 | valid_mae=2616.16
Epoch 002 | train_loss=0.18542 | valid_loss=0.44568 | valid_wmae=2588.37 | valid_mae=2581.41
Epoch 003 | train_loss=0.17628 | valid_loss=0.44448 | valid_wmae=2575.76 | valid_mae=2568.93
Epoch 004 | train_loss=0.17303 | valid_loss=0.44501 | valid_wmae=2571.67 | valid_mae=2563.49
Epoch 005 | train_loss=0.17178 | valid_loss=0.44294 | valid_wmae=2557.07 | valid_mae=2550.53
Epoch 006 | train_loss=0.17136 | valid_loss=0.44354 | valid_wmae=2561.12 | valid_mae=2553.59
Epoch 007 | train_loss=0.17123 | valid_loss=0.44541 | valid_wmae=2566.88 | valid_mae=2559.15
Epoch 008 | train_loss=0.17122 | valid_loss=0.44480 | valid_wmae=2564.90 | valid_mae=2559.38
Epoch 009 | train_loss=0.17123 | valid_loss=0.44641 | valid_wmae=2574.52 | valid_mae=2566.81
Epoch 010 | train_loss=0.1712

,run_name,run_id,kernel_size,lr,weight_decay,loss_name,best_valid_wmae,best_epoch
0,DLinear_TargetOnly_k13_lr0.001_wd0.0001_huber,f64b527211a24007812e0db8cdca4d76,13,0.001,0.0001,huber,2555.540876,11


In [95]:
BEST_DLINEAR_CONFIG = {
    "kernel_size": 13,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "huber",
    "final_epochs": 24,
}

In [96]:
# full train preprocessing for final test inference

final_base_preprocessor = WalmartBasePreprocessor()
final_base_preprocessor.fit(stores, features)

full_train_base = final_base_preprocessor.transform(train)

final_neural_preprocessor = WalmartNeuralPreprocessor()
final_neural_preprocessor.fit(full_train_base)

full_train_panel = final_neural_preprocessor.transform(full_train_base)

final_dataset_cols = final_neural_preprocessor.get_dataset_columns()

print("Full train panel shape:", full_train_panel.shape)
print("Full train date range:", full_train_panel["Date"].min(), "->", full_train_panel["Date"].max())
print("Full train unique dates:", full_train_panel["Date"].nunique())
print(final_dataset_cols)

Full train panel shape: (421570, 66)
Full train date range: 2010-02-05 00:00:00 -> 2012-10-26 00:00:00
Full train unique dates: 143
{'target_col': 'Weekly_Sales_scaled', 'series_col': 'series_id', 'static_cat_cols': ['Store_id', 'Dept_id', 'Type_id'], 'static_real_cols': ['Size_scaled'], 'known_future_real_cols': ['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'MarkDown1_scaled', 'MarkDown2_scaled', 'MarkDown3_scaled', 'MarkDown4_scaled', 'MarkDown5_scaled', 'total_markdown_scaled', 'abs_total_markdown_scaled', 'positive_markdown_sum_scaled', 'negative_markdown_sum_scaled', 'markdown_missing_count_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'has_markdown_signal', 'markdown_available_period', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']}


In [ ]:
test_dates = pd.DataFrame({
    "Date": sorted(pd.to_datetime(test["Date"]).unique())
})

test_pairs = test[["Store", "Dept"]].drop_duplicates().reset_index(drop=True)

# cross join: every test pair gets all 39 test dates (later we will merge IsHoliday from original test )
test_grid = test_pairs.merge(test_dates, how="cross")

# IsHoliday depends on Date so map it from original test
date_holiday = (
    test[["Date", "IsHoliday"]]
    .copy()
    .assign(Date=lambda x: pd.to_datetime(x["Date"]))
    .drop_duplicates()
)

assert date_holiday["Date"].nunique() == len(date_holiday)

test_grid = test_grid.merge(date_holiday, on="Date", how="left")

print("Original test rows:", len(test))
print("Full test grid rows:", len(test_grid))
print("Test pairs:", len(test_pairs))
print("Test dates:", len(test_dates))

assert test_grid["IsHoliday"].notna().all()
assert len(test_dates) == PREDICTION_LENGTH

Original test rows: 115064
Full test grid rows: 123591
Test pairs: 3169
Test dates: 39


In [98]:
full_test_grid_base = final_base_preprocessor.transform(test_grid)
full_test_grid_panel = final_neural_preprocessor.transform(full_test_grid_base)

print("Full test grid panel shape:", full_test_grid_panel.shape)
print("Full test grid date range:", full_test_grid_panel["Date"].min(), "->", full_test_grid_panel["Date"].max())
print("Full test grid unique dates:", full_test_grid_panel["Date"].nunique())

test_group_sizes = full_test_grid_panel.groupby("series_id").size()
assert (test_group_sizes == PREDICTION_LENGTH).all()

print("All test Store-Dept groups have 39 rows.")

Full test grid panel shape: (123591, 65)
Full test grid date range: 2012-11-02 00:00:00 -> 2013-07-26 00:00:00
Full test grid unique dates: 39
All test Store-Dept groups have 39 rows.


In [99]:
final_target_only_cols = {
    "target_col": final_dataset_cols["target_col"],
    "series_col": final_dataset_cols["series_col"],
    "static_cat_cols": [],
    "static_real_cols": [],
    "known_future_real_cols": ["IsHoliday"],
}

FINAL_BATCH_SIZE = 1024

final_train_dataset = WalmartPrecomputedTrainingWindowDataset(
    full_train_panel,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=final_target_only_cols["target_col"],
    series_col=final_target_only_cols["series_col"],
    static_cat_cols=final_target_only_cols["static_cat_cols"],
    static_real_cols=final_target_only_cols["static_real_cols"],
    known_future_real_cols=final_target_only_cols["known_future_real_cols"],
)

final_test_dataset = WalmartPrecomputedForecastWindowDataset(
    history_df=full_train_panel,
    future_df=full_test_grid_panel,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=final_target_only_cols["target_col"],
    series_col=final_target_only_cols["series_col"],
    static_cat_cols=final_target_only_cols["static_cat_cols"],
    static_real_cols=final_target_only_cols["static_real_cols"],
    known_future_real_cols=final_target_only_cols["known_future_real_cols"],
)

final_train_loader = FastTensorDataLoader(
    final_train_dataset.tensors,
    batch_size=FINAL_BATCH_SIZE,
    shuffle=True,
)

final_test_loader = FastTensorDataLoader(
    final_test_dataset.tensors,
    batch_size=FINAL_BATCH_SIZE,
    shuffle=False,
)

print("Final train windows:", len(final_train_dataset))
print("Final test forecast series:", len(final_test_dataset))
print("Final train batches:", len(final_train_loader))
print("Final test batches:", len(final_test_loader))

Final train windows: 149069
Final test forecast series: 3169
Final train batches: 146
Final test batches: 4


In [100]:
def train_final_dlinear_target_only(
    config: dict,
    train_loader,
    device,
):
    model = DLinear(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        kernel_size=config["kernel_size"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    epochs = config["final_epochs"]

    for epoch in range(1, epochs + 1):
        model.train()

        total_loss = 0.0
        total_items = 0

        for batch in train_loader:
            batch = move_batch_to_device(batch, device)

            optimizer.zero_grad()

            preds = model(batch["past_target"])
            target = batch["future_target"]

            loss = loss_fn(preds, target)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            n_items = target.numel()
            total_loss += loss.item() * n_items
            total_items += n_items

        train_loss = total_loss / total_items

        print(f"Epoch {epoch:03d} | final_train_loss={train_loss:.5f}")

    return model

In [101]:
@torch.no_grad()
def predict_dlinear_target_only(model, loader, device):
    model.eval()

    all_preds = []

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        preds_scaled = model(batch["past_target"])

        preds_original = (
            preds_scaled * batch["target_std"].unsqueeze(1)
            + batch["target_mean"].unsqueeze(1)
        )

        all_preds.append(preds_original.detach().cpu().numpy())

    return np.concatenate(all_preds, axis=0)

In [102]:
final_dlinear_model = train_final_dlinear_target_only(
    BEST_DLINEAR_CONFIG,
    final_train_loader,
    DEVICE,
)

test_preds_matrix = predict_dlinear_target_only(
    final_dlinear_model,
    final_test_loader,
    DEVICE,
)

print("Test prediction matrix shape:", test_preds_matrix.shape)
assert test_preds_matrix.shape == (len(final_test_dataset), PREDICTION_LENGTH)

Epoch 001 | final_train_loss=0.31407
Epoch 002 | final_train_loss=0.24015
Epoch 003 | final_train_loss=0.22960
Epoch 004 | final_train_loss=0.22554
Epoch 005 | final_train_loss=0.22402
Epoch 006 | final_train_loss=0.22352
Epoch 007 | final_train_loss=0.22338
Epoch 008 | final_train_loss=0.22336
Epoch 009 | final_train_loss=0.22335
Epoch 010 | final_train_loss=0.22335
Epoch 011 | final_train_loss=0.22336
Epoch 012 | final_train_loss=0.22337
Epoch 013 | final_train_loss=0.22337
Epoch 014 | final_train_loss=0.22338
Epoch 015 | final_train_loss=0.22338
Epoch 016 | final_train_loss=0.22337
Epoch 017 | final_train_loss=0.22339
Epoch 018 | final_train_loss=0.22337
Epoch 019 | final_train_loss=0.22338
Epoch 020 | final_train_loss=0.22341
Epoch 021 | final_train_loss=0.22339
Epoch 022 | final_train_loss=0.22339
Epoch 023 | final_train_loss=0.22339
Epoch 024 | final_train_loss=0.22339
Test prediction matrix shape: (3169, 39)


In [ ]:
test_index_df = final_test_dataset.get_future_index().reset_index(drop=True).copy()

test_preds_flat = test_preds_matrix.reshape(-1)

assert len(test_index_df) == len(test_preds_flat)

full_pred_df = test_index_df.copy()
full_pred_df["Weekly_Sales"] = test_preds_flat

# optional but usually sensible for submission..
full_pred_df["Weekly_Sales"] = full_pred_df["Weekly_Sales"].clip(lower=0)

print(full_pred_df.head())
print(full_pred_df["Weekly_Sales"].describe())

   Store  Dept       Date  Weekly_Sales
0     10     1 2012-11-02  60988.031250
1     10     1 2012-11-09  35257.902344
2     10     1 2012-11-16  29167.158203
3     10     1 2012-11-23  35994.664062
4     10     1 2012-11-30  37363.785156
count    123591.000000
mean      14643.471680
std       21369.892578
min           0.000000
25%        1407.797913
50%        6581.274414
75%       18536.495117
max      386121.500000
Name: Weekly_Sales, dtype: float64


In [104]:
test_keys = test[["Store", "Dept", "Date"]].copy()
test_keys["Date"] = pd.to_datetime(test_keys["Date"])

submission_df = test_keys.merge(
    full_pred_df,
    on=["Store", "Dept", "Date"],
    how="left",
)

assert len(submission_df) == len(test)
assert submission_df["Weekly_Sales"].notna().all()

submission_df["Id"] = (
    submission_df["Store"].astype(str)
    + "_"
    + submission_df["Dept"].astype(str)
    + "_"
    + submission_df["Date"].dt.strftime("%Y-%m-%d")
)

submission_df = submission_df[["Id", "Weekly_Sales"]]

print(submission_df.head())
print(submission_df.shape)

               Id  Weekly_Sales
0  1_1_2012-11-02  30560.611328
1  1_1_2012-11-09  19558.984375
2  1_1_2012-11-16  18501.179688
3  1_1_2012-11-23  19012.458984
4  1_1_2012-11-30  20660.041016
(115064, 2)


In [105]:
submission_name = (
    f"submission_dlinear_target_only_"
    f"k{BEST_DLINEAR_CONFIG['kernel_size']}_"
    f"{BEST_DLINEAR_CONFIG['loss_name']}_"
    f"epochs{BEST_DLINEAR_CONFIG['final_epochs']}.csv"
)

submission_path = repo_root / submission_name

submission_df.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)

Saved submission to: C:\Users\Myvari\Desktop\walmart-sales-forecasting\submission_dlinear_target_only_k13_huber_epochs24.csv


## DLinear-X

In [ ]:
set_seed(SEED)
full_feature_batch = next(iter(dlinearx_train_loader))

dlinearx_model = DLinearX(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    static_cat_cardinalities=static_cat_cardinalities,
    num_future_reals=num_future_reals,
    num_static_reals=num_static_reals,
    kernel_size=13,
    embedding_dim=8,
    hidden_dim=64,
    dropout=0.1,
).to(DEVICE)

full_feature_batch_device = move_batch_to_device(full_feature_batch, DEVICE)

with torch.no_grad():
    out = dlinearx_model(
        past_target=full_feature_batch_device["past_target"],
        future_known_reals=full_feature_batch_device["future_known_reals"],
        static_categoricals=full_feature_batch_device["static_categoricals"],
        static_reals=full_feature_batch_device["static_reals"],
    )

print("DLinear-X output shape:", tuple(out.shape))
assert out.shape == full_feature_batch_device["future_target"].shape

del dlinearx_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("DLinear-X shape test passed.")

DLinear-X output shape: (512, 39)
DLinear-X shape test passed.


In [ ]:
# DLINEARX_CONFIG = {
#     "model_type": "DLinearX",
#     "model_variant": "dlinearx",
#     "feature_set": "past_target_plus_future_covariates_plus_static",
#     "context_length": CONTEXT_LENGTH,
#     "prediction_length": PREDICTION_LENGTH,
#     "kernel_size": 25,
#     "embedding_dim": 8,
#     "hidden_dim": 64,
#     "dropout": 0.1,
#     "batch_size": BATCH_SIZE,
#     "learning_rate": 1e-3,
#     "weight_decay": 1e-4,
#     "epochs": 10,
#     "loss_function": "MSELoss",
#     "optimizer": "AdamW",
#     "num_future_reals": num_future_reals,
#     "num_static_reals": num_static_reals,
#     "num_static_categoricals": len(static_cat_cardinalities),
# }

set_seed(SEED)
DLINEARX_GRID = [
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 64,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 64,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "kernel_size": 7,
        "embedding_dim": 8,
        "hidden_dim": 64,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 64,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "mse",
    },
]

# dlinearx_model = DLinearX(
#     context_length=CONTEXT_LENGTH,
#     prediction_length=PREDICTION_LENGTH,
#     static_cat_cardinalities=static_cat_cardinalities,
#     num_future_reals=num_future_reals,
#     num_static_reals=num_static_reals,
#     kernel_size=DLINEARX_CONFIG["kernel_size"],
#     embedding_dim=DLINEARX_CONFIG["embedding_dim"],
#     hidden_dim=DLINEARX_CONFIG["hidden_dim"],
#     dropout=DLINEARX_CONFIG["dropout"],
# ).to(DEVICE)

# print(dlinearx_model)
# print("Trainable parameters:", count_parameters(dlinearx_model))

In [132]:
future_cols = dlinearx_calendar_cols["known_future_real_cols"]

train_fk = dlinearx_train_dataset.tensors["future_known_reals"]
valid_fk = dlinearx_valid_dataset.tensors["future_known_reals"]

cov_stats = pd.DataFrame({
    "feature": future_cols,
    "train_min": train_fk.amin(dim=(0, 1)).cpu().numpy(),
    "train_max": train_fk.amax(dim=(0, 1)).cpu().numpy(),
    "valid_min": valid_fk.amin(dim=(0, 1)).cpu().numpy(),
    "valid_max": valid_fk.amax(dim=(0, 1)).cpu().numpy(),
})

cov_stats["train_absmax"] = cov_stats[["train_min", "train_max"]].abs().max(axis=1)
cov_stats["valid_absmax"] = cov_stats[["valid_min", "valid_max"]].abs().max(axis=1)

display(
    cov_stats
    .sort_values("valid_absmax", ascending=False)
    .head(20)
)

,feature,train_min,train_max,valid_min,valid_max,train_absmax,valid_absmax
0,Temperature_scaled,-3.301547,2.076778,-2.798188,1.996938,3.301547,2.798188
1,Fuel_Price_scaled,-0.478972,2.257342,-0.352885,2.471911,2.257342,2.471911
3,Unemployment_scaled,-2.068945,3.078008,-2.252822,2.471696,3.078008,2.471696
5,Week_cos_scaled,-1.316503,1.420794,-1.316503,1.686592,1.420794,1.686592
2,CPI_scaled,-1.081326,1.361102,-1.026901,1.475428,1.361102,1.475428
4,Week_sin_scaled,-1.403510,1.317423,-1.162686,1.317423,1.403510,1.317423
6,IsHoliday,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000
7,IsSuperBowl,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000
9,IsThanksgiving,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000
10,IsChristmas,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000


In [133]:
display(cov_stats[cov_stats["feature"].str.lower().str.contains("markdown")])

,feature,train_min,train_max,valid_min,valid_max,train_absmax,valid_absmax


In [ ]:
dlinearx_results = []

EPOCHS_DLINEARX = 25

for i, config in enumerate(DLINEARX_GRID, start=1):
    run_name = (
        f"DLinearX_CalendarAligned_"
        f"k{config['kernel_size']}_"
        f"emb{config['embedding_dim']}_"
        f"h{config['hidden_dim']}_"
        f"drop{config['dropout']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"{config['loss_name']}"
    )

    print("=" * 80)
    print(f"Run {i}/{len(DLINEARX_GRID)}: {run_name}")
    print(config)

    set_seed(SEED + i)

    model = DLinearX(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=static_cat_cardinalities,
        num_future_reals=num_future_reals,
        num_static_reals=num_static_reals,
        kernel_size=config["kernel_size"],
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "DLinearX",
            "model_variant": "dlinearx",
            "feature_set": "past_target_plus_future_covariates_plus_static",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "kernel_size": config["kernel_size"],
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_DLINEARX,
            "optimizer": "AdamW",
            "validation_strategy": "calendar_aligned_39_weeks",
            "num_future_reals": num_future_reals,
            "num_static_reals": num_static_reals,
            "num_static_categoricals": len(static_cat_cardinalities),
            "train_windows": len(dlinearx_train_dataset),
            "valid_forecast_series": len(dlinearx_valid_dataset),
            "num_parameters": count_parameters(model),
        })

        mlflow.log_param("static_cat_cardinalities", str(static_cat_cardinalities))

        result = fit_model(
            model=model,
            train_loader=dlinearx_train_loader,
            valid_loader=dlinearx_valid_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            model_variant="dlinearx",
            holiday_feature_idx=DLINEARX_HOLIDAY_FEATURE_IDX,
            epochs=EPOCHS_DLINEARX,
        )

        mlflow.log_metric("best_valid_wmae", result["best_valid_wmae"])
        mlflow.log_metric("best_epoch", result["best_epoch"])

        dlinearx_results.append({
            "run_name": run_name,
            "run_id": run_id,
            **config,
            "best_valid_wmae": result["best_valid_wmae"],
            "best_epoch": result["best_epoch"],
            "num_parameters": count_parameters(result["model"]),
        })

        print("Best WMAE:", result["best_valid_wmae"])
        print("Best epoch:", result["best_epoch"])

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

dlinearx_results_df = (
    pd.DataFrame(dlinearx_results)
    .sort_values("best_valid_wmae")
    .reset_index(drop=True)
)

display(dlinearx_results_df)

Run 1/4: DLinearX_CalendarAligned_k13_emb8_h64_drop0.2_lr0.001_wd0.0001_huber
{'kernel_size': 13, 'embedding_dim': 8, 'hidden_dim': 64, 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'huber'}
Epoch 001 | train_loss=0.40724 | valid_loss=0.66669 | valid_wmae=4279.16 | valid_mae=3893.85
Epoch 002 | train_loss=0.32476 | valid_loss=0.64789 | valid_wmae=4151.98 | valid_mae=3782.05
Epoch 003 | train_loss=0.27456 | valid_loss=0.63371 | valid_wmae=4032.75 | valid_mae=3689.26
Epoch 004 | train_loss=0.24336 | valid_loss=0.62541 | valid_wmae=3947.15 | valid_mae=3626.33
Epoch 005 | train_loss=0.22145 | valid_loss=0.62031 | valid_wmae=3888.94 | valid_mae=3587.55
Epoch 006 | train_loss=0.20612 | valid_loss=0.61710 | valid_wmae=3848.01 | valid_mae=3555.92
Epoch 007 | train_loss=0.19492 | valid_loss=0.61396 | valid_wmae=3815.96 | valid_mae=3526.91
Epoch 008 | train_loss=0.18674 | valid_loss=0.61172 | valid_wmae=3791.15 | valid_mae=3499.34
Epoch 009 | train_loss=0.18039 | valid_loss=0

,run_name,run_id,kernel_size,embedding_dim,hidden_dim,dropout,lr,weight_decay,loss_name,best_valid_wmae,best_epoch,num_parameters
0,DLinearX_CalendarAligned_k13_emb8_h128_drop0.2...,888edb214cee4a5abc4bfeb6db68b0ff,13,8,128,0.2,0.0005,0.0001,huber,3450.116026,25,10055
1,DLinearX_CalendarAligned_k7_emb8_h64_drop0.1_l...,033a21a6abfe42cfb4ac629ac2093041,7,8,64,0.1,0.0010,0.0001,huber,3550.640396,25,7623
2,DLinearX_CalendarAligned_k13_emb8_h64_drop0.2_...,d0878b09e062441099aa4cc8cbf2b435,13,8,64,0.2,0.0010,0.0001,huber,3571.657970,25,7623
3,DLinearX_CalendarAligned_k13_emb8_h64_drop0.1_...,46c708c9c2bf4d82829addfddfc7308f,13,8,64,0.1,0.0010,0.0001,mse,3609.315042,25,7623


In [135]:
dlinearx_results_v1 = []

EPOCHS_DLINEARX_V1 = 50

DLINEARX_EXTRA_GRID = [
    # same best config, just longer training
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # same architecture, faster learning
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # slightly more regularized version
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # lower dropout in case 0.2 is slowing learning too much
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # stronger weight decay because DLinear-X has a correction head that can overfit
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-3,
        "loss_name": "huber",
    },

    # test whether larger correction head helps
    {
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 256,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
]

for i, config in enumerate(DLINEARX_EXTRA_GRID, start=1):
    run_name = (
        f"DLinearX_CalendarAligned_"
        f"k{config['kernel_size']}_"
        f"emb{config['embedding_dim']}_"
        f"h{config['hidden_dim']}_"
        f"drop{config['dropout']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"{config['loss_name']}"
    )

    print("=" * 80)
    print(f"Run {i}/{len(DLINEARX_EXTRA_GRID)}: {run_name}")
    print(config)

    set_seed(SEED + i)

    model = DLinearX(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=static_cat_cardinalities,
        num_future_reals=num_future_reals,
        num_static_reals=num_static_reals,
        kernel_size=config["kernel_size"],
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "DLinearX",
            "model_variant": "dlinearx",
            "feature_set": "past_target_plus_future_covariates_plus_static",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "kernel_size": config["kernel_size"],
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_DLINEARX_V1,
            "optimizer": "AdamW",
            "validation_strategy": "calendar_aligned_39_weeks",
            "num_future_reals": num_future_reals,
            "num_static_reals": num_static_reals,
            "num_static_categoricals": len(static_cat_cardinalities),
            "train_windows": len(dlinearx_train_dataset),
            "valid_forecast_series": len(dlinearx_valid_dataset),
            "num_parameters": count_parameters(model),
        })

        mlflow.log_param("static_cat_cardinalities", str(static_cat_cardinalities))

        result = fit_model(
            model=model,
            train_loader=dlinearx_train_loader,
            valid_loader=dlinearx_valid_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            model_variant="dlinearx",
            holiday_feature_idx=DLINEARX_HOLIDAY_FEATURE_IDX,
            epochs=EPOCHS_DLINEARX_V1,
        )

        mlflow.log_metric("best_valid_wmae", result["best_valid_wmae"])
        mlflow.log_metric("best_epoch", result["best_epoch"])

        dlinearx_results_v1.append({
            "run_name": run_name,
            "run_id": run_id,
            **config,
            "best_valid_wmae": result["best_valid_wmae"],
            "best_epoch": result["best_epoch"],
            "num_parameters": count_parameters(result["model"]),
        })

        print("Best WMAE:", result["best_valid_wmae"])
        print("Best epoch:", result["best_epoch"])

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

dlinearx_results_v1_df = (
    pd.DataFrame(dlinearx_results_v1)
    .sort_values("best_valid_wmae")
    .reset_index(drop=True)
)

display(dlinearx_results_v1_df)

Run 1/6: DLinearX_CalendarAligned_k13_emb8_h128_drop0.2_lr0.0005_wd0.0001_huber
{'kernel_size': 13, 'embedding_dim': 8, 'hidden_dim': 128, 'dropout': 0.2, 'lr': 0.0005, 'weight_decay': 0.0001, 'loss_name': 'huber'}
Epoch 001 | train_loss=0.42978 | valid_loss=0.68081 | valid_wmae=4336.69 | valid_mae=3973.97
Epoch 002 | train_loss=0.37808 | valid_loss=0.66404 | valid_wmae=4238.04 | valid_mae=3884.80
Epoch 003 | train_loss=0.33861 | valid_loss=0.65336 | valid_wmae=4168.91 | valid_mae=3819.96
Epoch 004 | train_loss=0.30778 | valid_loss=0.64269 | valid_wmae=4100.83 | valid_mae=3760.62
Epoch 005 | train_loss=0.28378 | valid_loss=0.63563 | valid_wmae=4044.82 | valid_mae=3713.85
Epoch 006 | train_loss=0.26479 | valid_loss=0.62944 | valid_wmae=3994.45 | valid_mae=3675.52
Epoch 007 | train_loss=0.24916 | valid_loss=0.62458 | valid_wmae=3953.19 | valid_mae=3644.68
Epoch 008 | train_loss=0.23689 | valid_loss=0.62129 | valid_wmae=3921.98 | valid_mae=3620.54
Epoch 009 | train_loss=0.22653 | valid_lo

,run_name,run_id,kernel_size,embedding_dim,hidden_dim,dropout,lr,weight_decay,loss_name,best_valid_wmae,best_epoch,num_parameters
0,DLinearX_CalendarAligned_k13_emb8_h128_drop0.3...,2730f02c8eb94a54ae5775ab0ba3b39c,13,8,128,0.3,0.0010,0.0001,huber,3436.093573,21,10055
1,DLinearX_CalendarAligned_k13_emb8_h128_drop0.2...,2d7ed1b37f2741a28d2243d58aaad1a3,13,8,128,0.2,0.0010,0.0010,huber,3479.318098,32,10055
2,DLinearX_CalendarAligned_k13_emb8_h128_drop0.2...,a7eaa62f23074f658ec8725d5381befa,13,8,128,0.2,0.0010,0.0001,huber,3503.892073,40,10055
3,DLinearX_CalendarAligned_k13_emb8_h128_drop0.1...,83be66c6dbdc406990c786acacad1156,13,8,128,0.1,0.0010,0.0001,huber,3520.543272,49,10055
4,DLinearX_CalendarAligned_k13_emb8_h128_drop0.2...,9e5f670509e54bb0b4bce8701df4fa71,13,8,128,0.2,0.0005,0.0001,huber,3550.386435,50,10055
5,DLinearX_CalendarAligned_k13_emb8_h256_drop0.3...,9d27500f3931424ca55886d8e56e6268,13,8,256,0.3,0.0005,0.0001,huber,3566.923458,50,14919


In [143]:
dlinearx_results_last39 = []

EPOCHS_DLINEARX_LAST39 = 25

DLINEARX_LAST39_GRID = [
    {
        "cols_name": "stable_no_markdowns",
        "data": last39_stable_data,
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "cols_name": "stable_no_markdowns",
        "data": last39_stable_data,
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "cols_name": "full_with_markdowns",
        "data": last39_full_data,
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "cols_name": "full_with_markdowns",
        "data": last39_full_data,
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
]

for i, config in enumerate(DLINEARX_LAST39_GRID, start=1):
    data = config["data"]

    static_cat_cardinalities_current = data["static_cat_cardinalities"]
    num_future_reals_current = data["num_future_reals"]
    num_static_reals_current = data["num_static_reals"]
    holiday_feature_idx_current = data["holiday_feature_idx"]

    run_name = (
        f"DLinearX_Last39_{config['cols_name']}_"
        f"k{config['kernel_size']}_"
        f"emb{config['embedding_dim']}_"
        f"h{config['hidden_dim']}_"
        f"drop{config['dropout']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"{config['loss_name']}"
    )

    print("=" * 80)
    print(f"Run {i}/{len(DLINEARX_LAST39_GRID)}: {run_name}")
    print({k: v for k, v in config.items() if k != "data"})

    set_seed(SEED + i)

    model = DLinearX(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=static_cat_cardinalities_current,
        num_future_reals=num_future_reals_current,
        num_static_reals=num_static_reals_current,
        kernel_size=config["kernel_size"],
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "DLinearX",
            "model_variant": "dlinearx",
            "feature_set": config["cols_name"],
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "kernel_size": config["kernel_size"],
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_DLINEARX_LAST39,
            "optimizer": "AdamW",
            "validation_strategy": "last_39_weeks",
            "num_future_reals": num_future_reals_current,
            "num_static_reals": num_static_reals_current,
            "num_static_categoricals": len(static_cat_cardinalities_current),
            "train_windows": len(data["train_dataset"]),
            "valid_forecast_series": len(data["valid_dataset"]),
            "num_parameters": count_parameters(model),
        })

        mlflow.log_param(
            "static_cat_cardinalities",
            str(static_cat_cardinalities_current),
        )

        result = fit_model(
            model=model,
            train_loader=data["train_loader"],
            valid_loader=data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            model_variant="dlinearx",
            holiday_feature_idx=holiday_feature_idx_current,
            epochs=EPOCHS_DLINEARX_LAST39,
        )

        mlflow.log_metric("best_valid_wmae", result["best_valid_wmae"])
        mlflow.log_metric("best_epoch", result["best_epoch"])

        dlinearx_results_last39.append({
            "run_name": run_name,
            "run_id": run_id,
            "feature_set": config["cols_name"],
            "kernel_size": config["kernel_size"],
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "dropout": config["dropout"],
            "lr": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "num_future_reals": num_future_reals_current,
            "best_valid_wmae": result["best_valid_wmae"],
            "best_epoch": result["best_epoch"],
            "num_parameters": count_parameters(result["model"]),
        })

        print("Best WMAE:", result["best_valid_wmae"])
        print("Best epoch:", result["best_epoch"])

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

dlinearx_results_last39_df = (
    pd.DataFrame(dlinearx_results_last39)
    .sort_values("best_valid_wmae")
    .reset_index(drop=True)
)

display(dlinearx_results_last39_df)

Run 1/4: DLinearX_Last39_stable_no_markdowns_k13_emb8_h128_drop0.2_lr0.0005_wd0.0001_huber
{'cols_name': 'stable_no_markdowns', 'kernel_size': 13, 'embedding_dim': 8, 'hidden_dim': 128, 'dropout': 0.2, 'lr': 0.0005, 'weight_decay': 0.0001, 'loss_name': 'huber'}
Epoch 001 | train_loss=0.30871 | valid_loss=0.41117 | valid_wmae=2504.18 | valid_mae=2510.34
Epoch 002 | train_loss=0.20648 | valid_loss=0.42031 | valid_wmae=2494.16 | valid_mae=2487.21
Epoch 003 | train_loss=0.18445 | valid_loss=0.42218 | valid_wmae=2497.14 | valid_mae=2489.96
Epoch 004 | train_loss=0.17553 | valid_loss=0.42163 | valid_wmae=2495.66 | valid_mae=2482.24
Epoch 005 | train_loss=0.17039 | valid_loss=0.42116 | valid_wmae=2489.41 | valid_mae=2474.98
Epoch 006 | train_loss=0.16687 | valid_loss=0.42092 | valid_wmae=2489.20 | valid_mae=2472.46
Epoch 007 | train_loss=0.16437 | valid_loss=0.41780 | valid_wmae=2467.01 | valid_mae=2451.27
Epoch 008 | train_loss=0.16249 | valid_loss=0.41920 | valid_wmae=2479.44 | valid_mae=24

,run_name,run_id,feature_set,kernel_size,embedding_dim,hidden_dim,dropout,lr,weight_decay,loss_name,num_future_reals,best_valid_wmae,best_epoch,num_parameters
0,DLinearX_Last39_stable_no_markdowns_k13_emb8_h...,c421909afd134b0398299be299ec7c86,stable_no_markdowns,13,8,128,0.2,0.0010,0.0001,huber,11,2403.404132,22,10055
1,DLinearX_Last39_stable_no_markdowns_k13_emb8_h...,695861f40f124096b590c9f31a82a0e5,stable_no_markdowns,13,8,128,0.2,0.0005,0.0001,huber,11,2421.133263,23,10055
2,DLinearX_Last39_full_with_markdowns_k13_emb8_h...,b26d5e2447ae41ff8d5181bc4d21a17b,full_with_markdowns,13,8,128,0.2,0.0005,0.0001,huber,28,2561.620422,9,12231
3,DLinearX_Last39_full_with_markdowns_k13_emb8_h...,06a01b3ec47249e681d39632c29616aa,full_with_markdowns,13,8,128,0.2,0.0010,0.0001,huber,28,2563.112578,2,12231


In [153]:
EPOCHS_DLINEARX_LAST39 = 25

DLINEARX_FLAGS_GRID = [
    {
        "cols_name": "markdown_flags_only",
        "data": last39_flags_data,
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "cols_name": "markdown_flags_only",
        "data": last39_flags_data,
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "cols_name": "full_with_markdowns",
        "data": last39_full_data,
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "cols_name": "stable_no_markdowns",
        "data": last39_stable_data,
        "kernel_size": 13,
        "embedding_dim": 8,
        "hidden_dim": 128,
        "dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
]

for i, config in enumerate(DLINEARX_FLAGS_GRID, start=1):
    data = config["data"]

    static_cat_cardinalities_current = data["static_cat_cardinalities"]
    num_future_reals_current = data["num_future_reals"]
    num_static_reals_current = data["num_static_reals"]
    holiday_feature_idx_current = data["holiday_feature_idx"]

    run_name = (
        f"DLinearX_Last39_{config['cols_name']}_"
        f"k{config['kernel_size']}_"
        f"emb{config['embedding_dim']}_"
        f"h{config['hidden_dim']}_"
        f"drop{config['dropout']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"{config['loss_name']}"
    )

    print("=" * 80)
    print(f"Run {i}/{len(DLINEARX_FLAGS_GRID)}: {run_name}")
    print({k: v for k, v in config.items() if k != "data"})

    set_seed(SEED + i)

    model = DLinearX(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=static_cat_cardinalities_current,
        num_future_reals=num_future_reals_current,
        num_static_reals=num_static_reals_current,
        kernel_size=config["kernel_size"],
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "DLinearX",
            "model_variant": "dlinearx",
            "feature_set": config["cols_name"],
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "kernel_size": config["kernel_size"],
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_DLINEARX_LAST39,
            "optimizer": "AdamW",
            "validation_strategy": "last_39_weeks",
            "num_future_reals": num_future_reals_current,
            "num_static_reals": num_static_reals_current,
            "num_static_categoricals": len(static_cat_cardinalities_current),
            "train_windows": len(data["train_dataset"]),
            "valid_forecast_series": len(data["valid_dataset"]),
            "num_parameters": count_parameters(model),
        })

        mlflow.log_param(
            "static_cat_cardinalities",
            str(static_cat_cardinalities_current),
        )

        result = fit_model(
            model=model,
            train_loader=data["train_loader"],
            valid_loader=data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            model_variant="dlinearx",
            holiday_feature_idx=holiday_feature_idx_current,
            epochs=EPOCHS_DLINEARX_LAST39,
        )

        mlflow.log_metric("best_valid_wmae", result["best_valid_wmae"])
        mlflow.log_metric("best_epoch", result["best_epoch"])

        dlinearx_results_last39.append({
            "run_name": run_name,
            "run_id": run_id,
            "feature_set": config["cols_name"],
            "kernel_size": config["kernel_size"],
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "dropout": config["dropout"],
            "lr": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "num_future_reals": num_future_reals_current,
            "best_valid_wmae": result["best_valid_wmae"],
            "best_epoch": result["best_epoch"],
            "num_parameters": count_parameters(result["model"]),
        })

        print("Best WMAE:", result["best_valid_wmae"])
        print("Best epoch:", result["best_epoch"])

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

dlinearx_results_last39_df = (
    pd.DataFrame(dlinearx_results_last39)
    .sort_values("best_valid_wmae")
    .reset_index(drop=True)
)

display(dlinearx_results_last39_df)

Run 1/1: DLinearX_Last39_stable_no_markdowns_k13_emb8_h128_drop0.3_lr0.001_wd0.0001_huber
{'cols_name': 'stable_no_markdowns', 'kernel_size': 13, 'embedding_dim': 8, 'hidden_dim': 128, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'huber'}
Epoch 001 | train_loss=0.25816 | valid_loss=0.42420 | valid_wmae=2504.08 | valid_mae=2501.70
Epoch 002 | train_loss=0.18070 | valid_loss=0.42380 | valid_wmae=2502.29 | valid_mae=2487.19
Epoch 003 | train_loss=0.16987 | valid_loss=0.42269 | valid_wmae=2491.62 | valid_mae=2478.17
Epoch 004 | train_loss=0.16490 | valid_loss=0.42209 | valid_wmae=2484.37 | valid_mae=2465.18
Epoch 005 | train_loss=0.16199 | valid_loss=0.42208 | valid_wmae=2484.53 | valid_mae=2462.58
Epoch 006 | train_loss=0.16029 | valid_loss=0.42060 | valid_wmae=2475.17 | valid_mae=2453.06
Epoch 007 | train_loss=0.15905 | valid_loss=0.41700 | valid_wmae=2450.98 | valid_mae=2431.33
Epoch 008 | train_loss=0.15812 | valid_loss=0.41796 | valid_wmae=2463.62 | valid_mae=2438

,run_name,run_id,feature_set,kernel_size,embedding_dim,hidden_dim,dropout,lr,weight_decay,loss_name,num_future_reals,best_valid_wmae,best_epoch,num_parameters
0,DLinearX_Last39_stable_no_markdowns_k13_emb8_h...,c421909afd134b0398299be299ec7c86,stable_no_markdowns,13,8,128,0.2,0.0010,0.0001,huber,11,2403.404132,22,10055
1,DLinearX_Last39_stable_no_markdowns_k13_emb8_h...,854a4f13307242a880ab59745be6a5e2,stable_no_markdowns,13,8,128,0.3,0.0010,0.0001,huber,11,2418.578399,20,10055
2,DLinearX_Last39_stable_no_markdowns_k13_emb8_h...,695861f40f124096b590c9f31a82a0e5,stable_no_markdowns,13,8,128,0.2,0.0005,0.0001,huber,11,2421.133263,23,10055
3,DLinearX_Last39_markdown_flags_only_k13_emb8_h...,39278f9f54de4e73a032f83c0cdd250e,markdown_flags_only,13,8,128,0.3,0.0010,0.0001,huber,18,2468.593841,4,10951
4,DLinearX_Last39_markdown_flags_only_k13_emb8_h...,744b6f0e9ac14f25b2da160dda70b15f,markdown_flags_only,13,8,128,0.2,0.0010,0.0001,huber,18,2476.000231,4,10951
5,DLinearX_Last39_full_with_markdowns_k13_emb8_h...,b26d5e2447ae41ff8d5181bc4d21a17b,full_with_markdowns,13,8,128,0.2,0.0005,0.0001,huber,28,2561.620422,9,12231
6,DLinearX_Last39_full_with_markdowns_k13_emb8_h...,06a01b3ec47249e681d39632c29616aa,full_with_markdowns,13,8,128,0.2,0.0010,0.0001,huber,28,2563.112578,2,12231
7,DLinearX_Last39_full_with_markdowns_k13_emb8_h...,f65714e170cc4a43b1235c748eb68e77,full_with_markdowns,13,8,128,0.3,0.0010,0.0001,huber,28,2570.305127,7,12231


In [ ]:
EXPERIMENT_NAME = "DLinear_Training"

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    output_format="pandas",
)

print("Total MLflow runs:", len(runs))

# convenience helpers
def col_or_na(df, col):
    if col in df.columns:
        return df[col]
    return pd.Series(pd.NA, index=df.index)

def first_non_null(df, cols):
    existing = [col for col in cols if col in df.columns]
    if not existing:
        return pd.Series(pd.NA, index=df.index)
    return df[existing].bfill(axis=1).iloc[:, 0]


ranking = runs.copy()

# Run name
ranking["run_name"] = first_non_null(
    ranking,
    ["tags.mlflow.runName", "run_name"],
)

# Validation strategy
ranking["validation_strategy"] = first_non_null(
    ranking,
    ["params.validation_strategy"],
).astype("string")

# metadata
ranking["model"] = first_non_null(ranking, ["params.model"])
ranking["model_variant"] = first_non_null(ranking, ["params.model_variant"])
ranking["feature_set"] = first_non_null(ranking, ["params.feature_set"])

# hyperparams
ranking["kernel_size"] = first_non_null(ranking, ["params.kernel_size"])
ranking["embedding_dim"] = first_non_null(ranking, ["params.embedding_dim"])
ranking["hidden_dim"] = first_non_null(ranking, ["params.hidden_dim"])
ranking["dropout"] = first_non_null(ranking, ["params.dropout"])
ranking["lr"] = first_non_null(ranking, ["params.learning_rate", "params.lr"])
ranking["weight_decay"] = first_non_null(ranking, ["params.weight_decay"])
ranking["loss_name"] = first_non_null(ranking, ["params.loss_name", "params.loss_function"])
ranking["num_future_reals"] = first_non_null(ranking, ["params.num_future_reals"])

ranking["calendar_wmae"] = first_non_null(
    ranking,
    [
        "metrics.calendar_best_valid_wmae",
        "metrics.best_valid_wmae",
    ],
)

ranking["calendar_best_epoch"] = first_non_null(
    ranking,
    [
        "metrics.calendar_best_epoch",
        "metrics.best_epoch",
    ],
)

ranking["last39_wmae"] = col_or_na(ranking, "metrics.best_valid_wmae")
ranking["last39_best_epoch"] = col_or_na(ranking, "metrics.best_epoch")

numeric_cols = [
    "calendar_wmae",
    "calendar_best_epoch",
    "last39_wmae",
    "last39_best_epoch",
    "kernel_size",
    "embedding_dim",
    "hidden_dim",
    "dropout",
    "lr",
    "weight_decay",
    "num_future_reals",
]

for col in numeric_cols:
    if col in ranking.columns:
        ranking[col] = pd.to_numeric(ranking[col], errors="coerce")

Total MLflow runs: 50


In [157]:
calendar_mask = (
    ranking["validation_strategy"].str.contains("calendar", case=False, na=False)
    |
    ranking["run_name"].astype("string").str.contains("Calendar", case=False, na=False)
    |
    ranking["calendar_wmae"].notna() & col_or_na(ranking, "metrics.calendar_best_valid_wmae").notna()
)

calendar_ranking_cols = [
    "run_name",
    "run_id",
    "validation_strategy",
    "model",
    "model_variant",
    "feature_set",
    "kernel_size",
    "embedding_dim",
    "hidden_dim",
    "dropout",
    "lr",
    "weight_decay",
    "loss_name",
    "num_future_reals",
    "calendar_wmae",
    "calendar_best_epoch",
]

calendar_ranking_cols = [
    col for col in calendar_ranking_cols
    if col in ranking.columns
]

calendar_ranking_df = (
    ranking.loc[calendar_mask, calendar_ranking_cols]
    .dropna(subset=["calendar_wmae"])
    .sort_values("calendar_wmae", ascending=True)
    .reset_index(drop=True)
)

print(calendar_ranking_df.head(20).to_markdown(index=False))

| run_name                                                               | run_id                           | validation_strategy       | model    | model_variant   | feature_set                                    |   kernel_size |   embedding_dim |   hidden_dim |   dropout |     lr |   weight_decay | loss_name   |   num_future_reals |   calendar_wmae |   calendar_best_epoch |
|:-----------------------------------------------------------------------|:---------------------------------|:--------------------------|:---------|:----------------|:-----------------------------------------------|--------------:|----------------:|-------------:|----------:|-------:|---------------:|:------------|-------------------:|----------------:|----------------------:|
| DLinearX_CalendarAligned_k13_emb8_h128_drop0.3_lr0.001_wd0.0001_huber  | 2730f02c8eb94a54ae5775ab0ba3b39c | calendar_aligned_39_weeks | DLinearX | dlinearx        | past_target_plus_future_covariates_plus_static |            13 |         

In [ ]:
last39_mask = (
    ranking["validation_strategy"].str.contains("last_39|last39", case=False, na=False)
    |
    ranking["run_name"].astype("string").str.contains("Last39|Last_39", case=False, na=False)
)

last39_mask = last39_mask & ~ranking["validation_strategy"].str.contains(
    "calendar",
    case=False,
    na=False,
)

last39_ranking_cols = [
    "run_name",
    "run_id",
    "validation_strategy",
    "model",
    "model_variant",
    "feature_set",
    "kernel_size",
    "embedding_dim",
    "hidden_dim",
    "dropout",
    "lr",
    "weight_decay",
    "loss_name",
    "num_future_reals",
    "last39_wmae",
    "last39_best_epoch",
]

last39_ranking_cols = [
    col for col in last39_ranking_cols
    if col in ranking.columns
]

last39_ranking_df = (
    ranking.loc[last39_mask, last39_ranking_cols]
    .dropna(subset=["last39_wmae"])
    .sort_values("last39_wmae", ascending=True)
    .reset_index(drop=True)
)

print(last39_ranking_df.head(20).to_markdown(index=False))

| run_name                                                                          | run_id                           | validation_strategy   | model    | model_variant   | feature_set         |   kernel_size |   embedding_dim |   hidden_dim |   dropout |     lr |   weight_decay | loss_name   |   num_future_reals |   last39_wmae |   last39_best_epoch |
|:----------------------------------------------------------------------------------|:---------------------------------|:----------------------|:---------|:----------------|:--------------------|--------------:|----------------:|-------------:|----------:|-------:|---------------:|:------------|-------------------:|--------------:|--------------------:|
| DLinearX_Last39_stable_no_markdowns_k13_emb8_h128_drop0.2_lr0.001_wd0.0001_huber  | c421909afd134b0398299be299ec7c86 | last_39_weeks         | DLinearX | dlinearx        | stable_no_markdowns |            13 |               8 |          128 |       0.2 | 0.001  |         0.0001 | huber

models with no MarkDowns: `k13_emb8_h128_drop0.3_lr0.001_wd0.0001_huber` seems to be performing the best overall, `k13_emb8_h128_drop0.2_lr0.0005_wd0.0001_huber` close second could benefit from more epochs